In [36]:
from IPython.display import display
import time
import hmac
import hashlib
import urllib.request
from urllib.parse import quote
import json
import pandas as pd
import numpy as np
from typing import Dict, Any, List, Optional

API_KEY = "9cc2976dc30ff6190b7b0485536cc483"
SECRET  = "61f2cb7a045b539c7a530af70a5be50cbff571f69e6dd8fbaa3dcc1fb8835ac8"

BASE_URL = "https://api.statiz.co.kr/baseballApi"

### 공통함수

In [3]:
# 공통 함수

def normalize_query(params: dict) -> str:
    safe = "-_.!~*'()"
    return "&".join(
        f"{quote(str(k), safe=safe)}={quote(str(params[k]), safe=safe)}"
        for k in sorted(params.keys())
        if params[k] is not None
    )


def make_signature(secret: str, payload: str) -> str:
    return hmac.new(
        secret.encode("utf-8"),
        payload.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()


def api_get(path: str, query: Dict[str, Any], timeout: int = 30, verbose: bool = False) -> Dict[str, Any]:
    method = "GET"
    normalized = normalize_query(query)
    timestamp = str(int(time.time()))
    payload = f"{method}|{path}|{normalized}|{timestamp}"
    signature = make_signature(SECRET, payload)

    url = f"{BASE_URL}/{path}"
    if normalized:
        url = f"{url}?{normalized}"

    headers = {
        "X-API-KEY": API_KEY,
        "X-TIMESTAMP": timestamp,
        "X-SIGNATURE": signature,
    }

    if verbose:
        print("URL:", url)
        print("Payload:", payload)

    req = urllib.request.Request(url, method=method, headers=headers)

    with urllib.request.urlopen(req, timeout=timeout) as resp:
        body = resp.read().decode("utf-8")

    data = json.loads(body)

    # 공통 실패 처리
    if isinstance(data, dict):
        result_cd = str(data.get("result_cd", ""))
        if result_cd and result_cd != "100":
            raise ValueError(f"API 실패: result_cd={result_cd}, result_msg={data.get('result_msg')}")

    return data


def numeric_key_rows(obj: Dict[str, Any]) -> List[Dict[str, Any]]:
    """
    응답이 {'0': {...}, '1': {...}, 'result_cd': '100', ...} 형태일 때
    숫자 key row들만 추출.
    """
    rows = []
    for k, v in obj.items():
        if str(k).isdigit() and isinstance(v, dict):
            rows.append(v)
    return rows


def to_df_from_list_or_numeric_dict(obj: Dict[str, Any], list_key: Optional[str] = None) -> pd.DataFrame:
    """
    응답이
    1) {'list': [...]} 형태이거나
    2) {'0': {...}, '1': {...}} 형태일 때
    DataFrame으로 변환.
    """
    if list_key is not None and list_key in obj and isinstance(obj[list_key], list):
        return pd.DataFrame(obj[list_key])

    rows = numeric_key_rows(obj)
    if rows:
        return pd.DataFrame(rows)

    return pd.DataFrame()

### 경기 일정 수집

In [4]:
def fetch_game_schedule(year: int, month: int, day: int, verbose: bool = False) -> pd.DataFrame:
    path = "prediction/gameSchedule"
    query = {
        "year": year,
        "month": month,
        "day": day
    }

    data = api_get(path, query, verbose=verbose)

    rows = []

    # 1) top-level에서 메타 키(result_cd 등) 제외하고, list value인 항목 수집
    if isinstance(data, dict):
        meta_keys = {"result_cd", "result_msg", "update_time"}
        for k, v in data.items():
            if k not in meta_keys and isinstance(v, list):
                rows.extend(v)

    df = pd.DataFrame(rows)

    if df.empty:
        if verbose:
            print("[fetch_game_schedule] rows is empty")
            print("raw keys:", list(data.keys()) if isinstance(data, dict) else type(data))
        return df

    # 컬럼명 표준화
    rename_map = {
        "s_no": "game_id",
        "gameDate": "game_date",
        "awayTeam": "away_team_code",
        "homeTeam": "home_team_code",
        "s_code": "stadium_code",
        "awaySP": "away_sp_id",
        "homeSP": "home_sp_id",
        "awaySPName": "away_sp_name",
        "homeSPName": "home_sp_name",
        "awayScore": "away_score",
        "homeScore": "home_score",
        "state": "game_state",
        "hm": "game_time",
    }
    df = df.rename(columns=rename_map)

    # game_date가 unix timestamp(초) 형태이므로 변환
    if "game_date" in df.columns:
        df["game_date"] = pd.to_datetime(df["game_date"], unit="s", errors="coerce")
    else:
        df["game_date"] = pd.to_datetime(f"{year}-{month:02d}-{day:02d}", errors="coerce")

    # season 보강
    if "season" not in df.columns:
        if "year" in df.columns:
            df["season"] = pd.to_numeric(df["year"], errors="coerce")
        else:
            df["season"] = year

    # home_win 생성
    if {"home_score", "away_score"}.issubset(df.columns):
        df["home_win"] = (
            pd.to_numeric(df["home_score"], errors="coerce") >
            pd.to_numeric(df["away_score"], errors="coerce")
        ).astype("Int64")
    else:
        df["home_win"] = pd.NA

    keep_cols = [
        "game_id",
        "game_date",
        "season",
        "home_team_code",
        "away_team_code",
        "stadium_code",
        "home_sp_id",
        "away_sp_id",
        "home_sp_name",
        "away_sp_name",
        "home_score",
        "away_score",
        "home_win",
        "game_state",
        "game_time",
        "weather",
        "temperature",
        "humidity",
        "windDirection",
        "windSpeed",
        "rainprobability",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]

    return df[keep_cols].copy()

In [5]:
# 여러날 일괄 수집

def collect_games(date_list: List[str], sleep_sec: float = 0.2) -> pd.DataFrame:
    all_dfs = []

    for dt in date_list:
        y, m, d = map(int, dt.split("-"))
        try:
            df = fetch_game_schedule(y, m, d)
            if not df.empty:
                all_dfs.append(df)
            print(f"[games] {dt}: {len(df)} rows")
        except Exception as e:
            print(f"[games] {dt}: ERROR -> {e}")
        time.sleep(sleep_sec)

    if all_dfs:
        out = pd.concat(all_dfs, ignore_index=True).drop_duplicates()
    else:
        out = pd.DataFrame()

    return out

### 경기 라인업 수집

In [6]:
def fetch_game_lineup(game_id: str, verbose: bool = False) -> pd.DataFrame:
    path = "prediction/gameLineup"
    query = {"s_no": str(game_id)}

    data = api_get(path, query, verbose=verbose)

    rows = []

    # 메타 키 제외, 팀코드 key 아래 list 수집
    if isinstance(data, dict):
        meta_keys = {"result_cd", "result_msg", "update_time"}
        for k, v in data.items():
            if k not in meta_keys and isinstance(v, list):
                rows.extend(v)

    df = pd.DataFrame(rows)

    if df.empty:
        if verbose:
            print("[fetch_game_lineup] rows is empty")
            if isinstance(data, dict):
                print("raw keys:", list(data.keys()))
        return df

    # 컬럼 표준화
    rename_map = {
        "s_no": "game_id",
        "p_no": "player_id",
        "t_code": "team_code",
        "p_name": "player_name",
        "starting": "is_starting",
        "battingOrder": "batting_order",
        "p_throw": "throws",
        "p_bat": "bats",
        "p_backNumber": "back_number",
    }
    df = df.rename(columns=rename_map)

    # game_id 문자열 통일
    df["game_id"] = df["game_id"].astype(str)

    # starting: Y/N -> 1/0
    if "is_starting" in df.columns:
        df["is_starting"] = df["is_starting"].map({"Y": 1, "N": 0}).fillna(df["is_starting"])

    # batting_order 숫자화
    if "batting_order" in df.columns:
        df["batting_order"] = pd.to_numeric(df["batting_order"], errors="coerce")

    # 선발투수 flag
    # position==1 이 투수라고 가정
    if "position" in df.columns:
        pos = pd.to_numeric(df["position"], errors="coerce")
        if "is_starting" in df.columns:
            st = pd.to_numeric(df["is_starting"], errors="coerce")
            df["is_starting_pitcher"] = ((pos == 1) & (st == 1)).astype(int)
        else:
            df["is_starting_pitcher"] = (pos == 1).astype(int)
    else:
        df["is_starting_pitcher"] = pd.NA

    # 중복 제거
    subset_cols = [c for c in ["game_id", "team_code", "player_id"] if c in df.columns]
    if subset_cols:
        df = df.drop_duplicates(subset=subset_cols).reset_index(drop=True)

    keep_cols = [
        "game_id",
        "game_date",
        "team_code",
        "player_id",
        "player_name",
        "is_starting",
        "lineupState",
        "batting_order",
        "position",
        "is_starting_pitcher",
        "throws",
        "bats",
        "back_number",
    ]
    keep_cols = [c for c in keep_cols if c in df.columns]

    return df[keep_cols].copy()

In [7]:
# games에서 lineup일괄 수집

def collect_lineups(games_df: pd.DataFrame, sleep_sec: float = 0.2) -> pd.DataFrame:
    if "game_id" not in games_df.columns:
        raise ValueError("games_df에 game_id 컬럼이 필요합니다.")

    all_dfs = []

    for game_id in games_df["game_id"].astype(str).tolist():
        try:
            df = fetch_game_lineup(game_id)

            if not df.empty:
                game_date = games_df.loc[
                    games_df["game_id"].astype(str) == game_id, "game_date"
                ]
                if not game_date.empty:
                    df["game_date"] = game_date.iloc[0]

                all_dfs.append(df)

            print(f"[lineup] {game_id}: {len(df)} rows")

        except Exception as e:
            print(f"[lineup] {game_id}: ERROR -> {e}")

        time.sleep(sleep_sec)  # 429 방지

    if all_dfs:
        out = pd.concat(all_dfs, ignore_index=True).drop_duplicates()
    else:
        out = pd.DataFrame()

    return out

### playerSeason: 타자/투수 시즌 누적 시트 수집

In [8]:
def convert_ip_baseball_to_decimal(x): ## 얘 다시
    """
    예: 66.1 -> 66.333..., 66.2 -> 66.666...
    문자열/숫자 모두 대응
    """
    if pd.isna(x):
        return pd.NA
    try:
        s = str(x)
        if "." not in s:
            return float(s)
        whole, frac = s.split(".")
        whole = int(whole)
        frac = int(frac)
        if frac == 0:
            return float(whole)
        elif frac == 1:
            return whole + 1/3
        elif frac == 2:
            return whole + 2/3
        else:
            return float(s)
    except Exception:
        return pd.NA


def fetch_player_season(player_id: str, verbose: bool = False) -> Dict[str, pd.DataFrame]:
    path = "prediction/playerSeason"
    query = {"p_no": str(player_id)}

    data = api_get(path, query, verbose=verbose)

    basic_df   = pd.DataFrame(data.get("basic", {}).get("list", []))
    deepen_df  = pd.DataFrame(data.get("deepen", {}).get("list", []))
    field_df   = pd.DataFrame(data.get("fielding", {}).get("list", []))

    # year 기준 merge
    if not basic_df.empty and not deepen_df.empty and "year" in basic_df.columns and "year" in deepen_df.columns:
        merged = basic_df.merge(deepen_df, on="year", how="left", suffixes=("", "_deepen"))
    else:
        merged = basic_df.copy()

    if not merged.empty:
        merged["player_id"] = str(player_id)

    # 투수/타자 구분: 보통 p_position == 1 이면 투수
    is_pitcher = False
    if not merged.empty and "p_position" in merged.columns:
        vals = merged["p_position"].dropna().astype(str).unique().tolist()
        is_pitcher = ("1" in vals)

    # 공통 컬럼명 표준화
    if "t_code" in merged.columns:
        merged = merged.rename(columns={"t_code": "team_code"})

    # IP 변환
    if "IP" in merged.columns:
        merged["IP_decimal"] = merged["IP"].apply(convert_ip_baseball_to_decimal)

    if is_pitcher:
        keep_cols = [
            "player_id", "year", "team_code",
            "GS", "IP", "IP_decimal", "ERA", "FIP", "WHIP",
            "K9", "BB9", "HR9", "KBB", "QS"
        ]
        keep_cols = [c for c in keep_cols if c in merged.columns]
        pitcher_df = merged[keep_cols].copy()
        hitter_df = pd.DataFrame()
    else:
        keep_cols = [
            "player_id", "year", "team_code",
            "PA", "AB", "H", "HR", "RBI", "BB", "SO",
            "AVG", "OBP", "SLG", "OPS", "wOBA", "wRCplus", "BBK"
        ]
        keep_cols = [c for c in keep_cols if c in merged.columns]
        hitter_df = merged[keep_cols].copy()
        pitcher_df = pd.DataFrame()

    return {
        "hitter": hitter_df,
        "pitcher": pitcher_df,
        "raw_basic": basic_df,
        "raw_deepen": deepen_df,
        "raw_fielding": field_df
    }

In [9]:
# 일괄수집

def collect_player_season(player_ids: List[str], snapshot_date: str, sleep_sec: float = 0.2):
    hitter_list = []
    pitcher_list = []

    for pid in map(str, player_ids):
        try:
            res = fetch_player_season(pid)
            if not res["hitter"].empty:
                df = res["hitter"].copy()
                df["snapshot_date"] = snapshot_date
                hitter_list.append(df)
            if not res["pitcher"].empty:
                df = res["pitcher"].copy()
                df["snapshot_date"] = snapshot_date
                pitcher_list.append(df)
            print(f"[playerSeason] {pid}: OK")
        except Exception as e:
            print(f"[playerSeason] {pid}: ERROR -> {e}")
        time.sleep(sleep_sec)

    hitter_out = pd.concat(hitter_list, ignore_index=True) if hitter_list else pd.DataFrame()
    pitcher_out = pd.concat(pitcher_list, ignore_index=True) if pitcher_list else pd.DataFrame()

    return hitter_out, pitcher_out

### playerRoster: 날짜별 팀 로스터 수집 (player_id list 확보용)

In [10]:
def fetch_player_roster(date: str, code: str, t_code: str, verbose: bool = False) -> pd.DataFrame:
    path = "prediction/playerRoster"
    query = {
        "date": date,
        "code": str(code),
        "t_code": str(t_code)
    }

    data = api_get(path, query, verbose=verbose)
    df = to_df_from_list_or_numeric_dict(data, list_key="list")

    if df.empty:
        return df

    rename_map = {
        "name": "player_name",
        "p_no": "player_id",
        "t_code": "team_code",
        "pj_date": "join_date"
    }
    df = df.rename(columns=rename_map)
    df["snapshot_date"] = date

    keep_cols = ["snapshot_date", "team_code", "player_id", "player_name", "join_date"]
    keep_cols = [c for c in keep_cols if c in df.columns]

    return df[keep_cols].copy()

In [11]:
# 일괄 수집

def collect_rosters(date: str, team_codes: List[str], code: str = "1", sleep_sec: float = 0.2) -> pd.DataFrame:
    all_dfs = []

    for t_code in map(str, team_codes):
        try:
            df = fetch_player_roster(date=date, code=code, t_code=t_code)
            if not df.empty:
                all_dfs.append(df)
            print(f"[roster] {date} team={t_code}: {len(df)} rows")
        except Exception as e:
            print(f"[roster] {date} team={t_code}: ERROR -> {e}")
        time.sleep(sleep_sec)

    if all_dfs:
        out = pd.concat(all_dfs, ignore_index=True).drop_duplicates()
    else:
        out = pd.DataFrame()

    return out

### playerDay: 타자/투수 일별 시트 수집 
- query 부분의 파라미터 명을 문서에 맞게 바꿀 것
> 보류

In [12]:
def fetch_player_day(player_id: str,
                     start_date: Optional[str] = None,
                     end_date: Optional[str] = None,
                     year: Optional[int] = None,
                     verbose: bool = False) -> Dict[str, pd.DataFrame]:
    """
    주의:
    playerDay의 실제 query 파라미터명(start_date/end_date 또는 year 등)은
    문서에 맞게 수정 필요할 수 있음.
    """
    path = "prediction/playerDay"

    query = {"p_no": str(player_id)}
    if start_date is not None:
        query["start_date"] = start_date
        query["startDate"] = start_date
    if end_date is not None:
        query["end_date"] = end_date
        query["endDate"] = end_date
    if year is not None:
        query["year"] = int(year)

    data = api_get(path, query, verbose=verbose)

    # 구조 방어적으로 파싱
    df = to_df_from_list_or_numeric_dict(data, list_key="list")
    if df.empty and "data" in data and isinstance(data["data"], dict):
        df = to_df_from_list_or_numeric_dict(data["data"], list_key="list")

    if df.empty:
        return {"hitter": pd.DataFrame(), "pitcher": pd.DataFrame(), "raw": df}

    # 공통 표준화
    rename_map = {
        "s_no": "game_id",
        "p_no": "player_id",
        "t_code": "team_code",
        "battingOrder": "batting_order"
    }
    df = df.rename(columns=rename_map)

    if "gameDate" in df.columns:
        df = df.rename(columns={"gameDate": "game_date"})

    if "game_date" in df.columns:
        raw_game_date = df["game_date"].astype(str).str.strip()
        numeric_game_date = pd.to_numeric(raw_game_date, errors="coerce")
        parsed_game_date = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

        numeric_mask = numeric_game_date.notna()
        if numeric_mask.any():
            unit = "ms" if numeric_game_date[numeric_mask].ge(10**12).any() else "s"
            parsed_game_date.loc[numeric_mask] = pd.to_datetime(
                numeric_game_date.loc[numeric_mask], unit=unit, errors="coerce"
            )

        text_mask = ~numeric_mask & raw_game_date.ne("") & raw_game_date.ne("nan")
        if text_mask.any():
            text_values = raw_game_date.loc[text_mask]
            parsed_text = pd.to_datetime(text_values, format="%Y-%m-%d %H:%M:%S", errors="coerce")
            parsed_text = parsed_text.fillna(
                pd.to_datetime(text_values, format="%Y-%m-%d", errors="coerce")
            )
            parsed_game_date.loc[text_mask] = parsed_text

        df["game_date"] = parsed_game_date

    # hitter / pitcher 분리
    # GS/IP/NP 존재하면 투수 가능성 높음
    pitcher_cols_signal = {"GS", "IP", "NP", "ERA", "WHIP"}
    hitter_cols_signal  = {"PA", "OBP", "SLG", "OPS", "batting_order"}

    pitcher_df = pd.DataFrame()
    hitter_df = pd.DataFrame()

    if len(pitcher_cols_signal.intersection(df.columns)) >= 2:
        keep_cols = [
            "game_id", "game_date", "player_id", "team_code",
            "GS", "IP", "ER", "H", "HR", "BB", "SO", "NP", "ERA", "WHIP"
        ]
        keep_cols = [c for c in keep_cols if c in df.columns]
        pitcher_df = df[keep_cols].copy()
        for col in ["IP", "ER", "H", "HR", "BB", "SO", "NP", "ERA", "WHIP"]:
            if col in pitcher_df.columns:
                pitcher_df[col] = pd.to_numeric(pitcher_df[col], errors="coerce")

        if "IP" in pitcher_df.columns:
            valid_pitching_row = pitcher_df["IP"].notna()
            if "NP" in pitcher_df.columns:
                valid_pitching_row = valid_pitching_row | pitcher_df["NP"].ge(10)
            if "ERA" in pitcher_df.columns:
                valid_pitching_row = valid_pitching_row | pitcher_df["ERA"].notna()
            if "WHIP" in pitcher_df.columns:
                valid_pitching_row = valid_pitching_row | pitcher_df["WHIP"].notna()
            pitcher_df = pitcher_df.loc[valid_pitching_row].copy()

    if len(hitter_cols_signal.intersection(df.columns)) >= 2:
        keep_cols = [
            "game_id", "game_date", "player_id", "team_code",
            "PA", "AB", "H", "HR", "RBI", "BB", "SO", "OBP", "SLG", "OPS", "batting_order"
        ]
        keep_cols = [c for c in keep_cols if c in df.columns]
        hitter_df = df[keep_cols].copy()

    return {"hitter": hitter_df, "pitcher": pitcher_df, "raw": df}

In [13]:
# 일괄 수집

def collect_player_day(player_ids: List[str],
                       start_date: Optional[str] = None,
                       end_date: Optional[str] = None,
                       year: Optional[int] = None,
                       sleep_sec: float = 0.2):
    hitter_list = []
    pitcher_list = []

    for pid in map(str, player_ids):
        try:
            res = fetch_player_day(
                player_id=pid,
                start_date=start_date,
                end_date=end_date,
                year=year
            )
            if not res["hitter"].empty:
                hitter_list.append(res["hitter"])
            if not res["pitcher"].empty:
                pitcher_list.append(res["pitcher"])
            print(f"[playerDay] {pid}: OK")
        except Exception as e:
            print(f"[playerDay] {pid}: ERROR -> {e}")
        time.sleep(sleep_sec)

    hitter_out = pd.concat(hitter_list, ignore_index=True) if hitter_list else pd.DataFrame()
    pitcher_out = pd.concat(pitcher_list, ignore_index=True) if pitcher_list else pd.DataFrame()

    return hitter_out, pitcher_out

## 실행

In [14]:
# 예시
date_list = ["2023-04-11", "2023-04-22", "2023-04-23"]
games_df = collect_games(date_list)
display(games_df.head())

[games] 2023-04-11: 5 rows
[games] 2023-04-22: 5 rows
[games] 2023-04-23: 5 rows


,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_score,home_win,game_state,game_time,weather,temperature,humidity,windDirection,windSpeed,rainprobability
0,20230041,2023-04-11 09:30:00,2023,6002,10001,1001,14770,11379,최승용,최원태,...,4,1,3,18:30:00,10400,13.0,84,21100,3.1,None
1,20230042,2023-04-11 09:30:00,2023,1001,9002,7003,14113,14581,원태인,오원석,...,5,0,3,18:30:00,10400,20.2,59,20900,1.8,None
2,20230043,2023-04-11 09:30:00,2023,2002,7002,6001,10058,14619,양현종,남지민,...,5,0,3,18:30:00,10400,16.9,92,21300,2.2,None
3,20230044,2023-04-11 09:30:00,2023,3001,5002,8001,15011,15486,반즈,박명근,...,5,1,3,18:30:00,10500,18.4,70,20900,3.4,None
4,20230045,2023-04-11 09:30:00,2023,11001,12001,8005,13085,15542,신민혁,슐서,...,0,1,3,18:30:00,10400,18.3,76,21500,1.0,None


In [15]:
games_df_clean = games_df[
    (games_df["game_state"] == 3) &
    (games_df["home_score"].notna()) &
    (games_df["away_score"].notna())
].copy()

In [16]:
test_lineup = fetch_game_lineup("20230092", verbose=True)
display(test_lineup)
print(test_lineup.shape)

URL: https://api.statiz.co.kr/baseballApi/prediction/gameLineup?s_no=20230092
Payload: GET|prediction/gameLineup|s_no=20230092|1775625956


,game_id,team_code,player_id,player_name,is_starting,lineupState,batting_order,position,is_starting_pitcher,throws,bats,back_number
0,20230092,9002,14888,추신수,1,Y,1.0,10,0,3,2,17
1,20230092,9002,14587,최지훈,1,Y,2.0,8,0,1,2,54
2,20230092,9002,10106,최정,1,Y,3.0,5,0,1,1,14
3,20230092,9002,15530,에레디아,1,Y,4.0,7,0,3,1,27
4,20230092,9002,10815,한유섬,1,Y,5.0,9,0,1,2,35
5,20230092,9002,10182,최주환,1,N,6.0,4,0,1,2,53
6,20230092,9002,10636,김성현,1,Y,7.0,6,0,1,1,6
7,20230092,9002,14699,전의산,1,N,8.0,3,0,1,2,56
8,20230092,9002,10707,김민식,1,Y,9.0,2,0,1,2,24
9,20230092,9002,15528,맥카티,1,N,NaN,1,1,3,2,33


(20, 12)


In [17]:
lineup_df = collect_lineups(games_df_clean)
display(lineup_df.head())
print(lineup_df.shape)

[lineup] 20230041: 20 rows
[lineup] 20230042: 20 rows
[lineup] 20230043: 20 rows
[lineup] 20230044: 20 rows
[lineup] 20230045: 20 rows
[lineup] 20230091: 20 rows
[lineup] 20230092: 20 rows
[lineup] 20230093: 20 rows
[lineup] 20230094: 20 rows
[lineup] 20230095: 20 rows
[lineup] 20230096: 20 rows
[lineup] 20230097: 20 rows
[lineup] 20230098: 20 rows
[lineup] 20230099: 20 rows
[lineup] 20230100: 20 rows


,game_id,team_code,player_id,player_name,is_starting,lineupState,batting_order,position,is_starting_pitcher,throws,bats,back_number,game_date
0,20230041,6002,10195,정수빈,1,Y,1.0,8,0,3,2,31,2023-04-11 09:30:00
1,20230041,6002,10184,허경민,1,Y,2.0,5,0,1,1,13,2023-04-11 09:30:00
2,20230041,6002,11213,양석환,1,Y,3.0,3,0,1,1,53,2023-04-11 09:30:00
3,20230041,6002,10643,김재환,1,Y,4.0,7,0,1,2,32,2023-04-11 09:30:00
4,20230041,6002,10165,양의지,1,Y,5.0,2,0,1,1,25,2023-04-11 09:30:00


(300, 13)


In [18]:
display(lineup_df.head(20))

,game_id,team_code,player_id,player_name,is_starting,lineupState,batting_order,position,is_starting_pitcher,throws,bats,back_number,game_date
0,20230041,6002,10195,정수빈,1,Y,1.0,8,0,3,2,31,2023-04-11 09:30:00
1,20230041,6002,10184,허경민,1,Y,2.0,5,0,1,1,13,2023-04-11 09:30:00
2,20230041,6002,11213,양석환,1,Y,3.0,3,0,1,1,53,2023-04-11 09:30:00
3,20230041,6002,10643,김재환,1,Y,4.0,7,0,1,2,32,2023-04-11 09:30:00
4,20230041,6002,10165,양의지,1,Y,5.0,2,0,1,1,25,2023-04-11 09:30:00
5,20230041,6002,15539,로하스,1,Y,6.0,10,0,1,2,11,2023-04-11 09:30:00
6,20230041,6002,11087,강승호,1,Y,7.0,4,0,1,1,23,2023-04-11 09:30:00
7,20230041,6002,14172,송승환,1,N,8.0,9,0,1,1,8,2023-04-11 09:30:00
8,20230041,6002,12894,이유찬,1,Y,9.0,6,0,1,1,7,2023-04-11 09:30:00
9,20230041,6002,14770,최승용,1,N,NaN,1,1,3,2,64,2023-04-11 09:30:00


In [19]:
team_codes = ["1001", "2002", "3001", "5002", "6002", "7002", "9002", "10001", "11001", "12001"]
#               삼성,    KIA,   롯데,     LG,   두산,   한화,    SSG,    키움,      NC,     KT

In [20]:
roster_df = collect_rosters(date="2023-04-22", team_codes=team_codes, code="1") # date는 변동할 것. 
display(roster_df.head())

[roster] 2023-04-22 team=1001: 28 rows
[roster] 2023-04-22 team=2002: 28 rows
[roster] 2023-04-22 team=3001: 28 rows
[roster] 2023-04-22 team=5002: 28 rows
[roster] 2023-04-22 team=6002: 28 rows
[roster] 2023-04-22 team=7002: 28 rows
[roster] 2023-04-22 team=9002: 28 rows
[roster] 2023-04-22 team=10001: 28 rows
[roster] 2023-04-22 team=11001: 28 rows
[roster] 2023-04-22 team=12001: 28 rows


,snapshot_date,team_code,player_id,player_name,join_date
0,2023-04-22,1001,10892,구자욱,2023-04-01
1,2023-04-22,1001,10363,백정현,2023-04-05
2,2023-04-22,1001,11404,장필준,2023-04-21
3,2023-04-22,1001,12530,최충연,2023-04-01
4,2023-04-22,1001,10685,김대우,2023-04-01


In [ ]:
'''
player_ids = roster_df["player_id"].astype(str).unique().tolist()
season_hit_df, season_pit_df = collect_player_season(player_ids, snapshot_date="2023-04-22")

target_year = pd.to_datetime(date_list[0]).year

season_hit_df["year"] = pd.to_numeric(season_hit_df["year"], errors="coerce")
season_pit_df["year"] = pd.to_numeric(season_pit_df["year"], errors="coerce")

season_hit_df = season_hit_df[season_hit_df["year"] == target_year].copy()
season_pit_df = season_pit_df[season_pit_df["year"] == target_year].copy()

display(season_hit_df.head())
display(season_pit_df.head())'''

[playerSeason] 10892: OK
[playerSeason] 10363: OK
[playerSeason] 11404: OK
[playerSeason] 12530: OK
[playerSeason] 10685: OK
[playerSeason] 10180: OK
[playerSeason] 10523: OK
[playerSeason] 12885: OK
[playerSeason] 10527: OK
[playerSeason] 10232: OK
[playerSeason] 13240: OK
[playerSeason] 13244: OK
[playerSeason] 11200: OK
[playerSeason] 14113: OK
[playerSeason] 10367: OK
[playerSeason] 12562: OK
[playerSeason] 14617: OK
[playerSeason] 14618: OK
[playerSeason] 11169: OK
[playerSeason] 12936: OK
[playerSeason] 10400: OK
[playerSeason] 14757: OK
[playerSeason] 14798: OK
[playerSeason] 14799: OK
[playerSeason] 12988: OK
[playerSeason] 15034: OK
[playerSeason] 14117: OK
[playerSeason] 13164: OK
[playerSeason] 10058: OK
[playerSeason] 10008: OK
[playerSeason] 11119: OK
[playerSeason] 10909: OK
[playerSeason] 10344: OK
[playerSeason] 11233: OK
[playerSeason] 12537: OK
[playerSeason] 11261: OK
[playerSeason] 11303: OK
[playerSeason] 11306: OK
[playerSeason] 11099: OK
[playerSeason] 11298: OK


KeyboardInterrupt: 

In [ ]:
'''cols = ["ERA", "FIP", "WHIP", "KBB"]
season_pit_df[cols] = season_pit_df[cols].replace([np.inf, -np.inf], np.nan).fillna(99.99)
season_pit_df.isna().sum()
'''

player_id        0
year             0
team_code        0
GS               0
IP               0
IP_decimal       0
ERA              0
FIP              0
WHIP             0
K9               0
BB9              0
HR9              0
KBB              0
QS               0
snapshot_date    0
dtype: int64

In [ ]:
# 보류
'''
day_hit_df, day_pit_df = collect_player_day(
    player_ids=player_ids,
    start_date="2025-03-28",
    end_date="2025-08-31"
)
display(day_hit_df.head())
display(day_pit_df.head())
'''

[playerDay] 10475: OK
[playerDay] 10749: OK
[playerDay] 10652: OK
[playerDay] 12697: OK
[playerDay] 13009: OK
[playerDay] 13056: OK
[playerDay] 10187: OK
[playerDay] 13109: OK
[playerDay] 12546: OK
[playerDay] 14137: OK
[playerDay] 14143: OK
[playerDay] 13112: OK
[playerDay] 12908: OK
[playerDay] 11190: OK
[playerDay] 11166: OK
[playerDay] 13102: OK
[playerDay] 13103: OK
[playerDay] 15486: OK
[playerDay] 15532: OK
[playerDay] 10387: OK
[playerDay] 15084: OK
[playerDay] 14140: OK
[playerDay] 14783: OK
[playerDay] 15081: OK
[playerDay] 11164: OK
[playerDay] 14846: OK
[playerDay] 16274: OK
[playerDay] 16286: OK
[playerDay] 11137: OK
[playerDay] 11172: OK
[playerDay] 13080: OK
[playerDay] 13081: OK
[playerDay] 13085: OK
[playerDay] 12897: OK
[playerDay] 11355: OK
[playerDay] 12560: OK
[playerDay] 14764: OK
[playerDay] 14765: OK
[playerDay] 10217: OK
[playerDay] 14220: OK
[playerDay] 10261: OK
[playerDay] 13082: OK
[playerDay] 14156: OK
[playerDay] 10891: OK
[playerDay] 14661: OK
[playerDay

,game_id,game_date,player_id,team_code,PA,AB,H,HR,RBI,BB,SO,OBP,SLG,OPS,batting_order
0,20260001.0,NaT,10475,5002,3,3,0,0,0,0,0,0.00,0.00,0.00,7
1,20260006.0,NaT,10475,5002,4,4,1,0,1,0,1,0.25,0.50,0.75,7
2,20260011.0,NaT,10475,5002,4,4,0,0,1,0,0,0.00,0.00,0.00,7
3,20260016.0,NaT,10475,5002,5,4,0,0,1,0,1,0.20,0.00,0.20,6
4,NaN,NaT,10749,5002,NaN,4,1,0,NaN,1,2,0.40,0.25,0.65,NaN


,game_id,game_date,player_id,team_code,GS,H,HR,BB,SO,NP,IP,ER,ERA,WHIP
0,20260001.0,NaT,10475,5002,1,0,0,0,0,7,NaN,NaN,NaN,NaN
1,20260006.0,NaT,10475,5002,1,1,0,0,1,19,NaN,NaN,NaN,NaN
2,20260011.0,NaT,10475,5002,1,0,0,0,0,9,NaN,NaN,NaN,NaN
3,20260016.0,NaT,10475,5002,1,0,0,0,1,14,NaN,NaN,NaN,NaN
4,NaN,NaT,10749,5002,0,1,0,1,2,20,1.0,0.0,0.0,2.0


### 경기단위 df 생성

In [ ]:
# 전처리 보조 함수

def prepare_games_for_modeling(games_df: pd.DataFrame) -> pd.DataFrame:
    df = games_df.copy()

    # 정상 경기만
    df = df[
        (df["game_state"] == 3) &
        (df["home_score"].notna()) &
        (df["away_score"].notna())
    ].copy()

    df["game_id"] = df["game_id"].astype(str)
    df["home_team_code"] = df["home_team_code"].astype(str)
    df["away_team_code"] = df["away_team_code"].astype(str)

    if "home_win" not in df.columns:
        df["home_win"] = (df["home_score"] > df["away_score"]).astype(int)

    return df


def prepare_lineup(lineup_df: pd.DataFrame) -> pd.DataFrame:
    df = lineup_df.copy()

    df["game_id"] = df["game_id"].astype(str)
    df["team_code"] = df["team_code"].astype(str)
    df["player_id"] = df["player_id"].astype(str)

    if "batting_order" in df.columns:
        df["batting_order"] = pd.to_numeric(df["batting_order"], errors="coerce")

    if "position" in df.columns:
        df["position"] = pd.to_numeric(df["position"], errors="coerce")

    if "is_starting" in df.columns:
        df["is_starting"] = pd.to_numeric(df["is_starting"], errors="coerce")

    if "is_starting_pitcher" in df.columns:
        df["is_starting_pitcher"] = pd.to_numeric(df["is_starting_pitcher"], errors="coerce")

    return df


def prepare_season_hitters(season_hit_df: pd.DataFrame) -> pd.DataFrame:
    df = season_hit_df.copy()

    df["player_id"] = df["player_id"].astype(str)
    df["team_code"] = df["team_code"].astype(str)

    num_cols = ["PA", "OBP", "SLG", "OPS", "wOBA", "wRCplus", "BBK"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df


def prepare_season_pitchers(season_pit_df: pd.DataFrame) -> pd.DataFrame:
    df = season_pit_df.copy()

    df["player_id"] = df["player_id"].astype(str)
    df["team_code"] = df["team_code"].astype(str)

    num_cols = ["GS", "IP", "IP_decimal", "ERA", "FIP", "WHIP", "K9", "BB9", "HR9", "KBB", "QS"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

In [ ]:
# 타자 feature 생성

def build_hitter_team_features(lineup_df: pd.DataFrame, season_hit_df: pd.DataFrame) -> pd.DataFrame:
    # 타자만: 선발투수 제외, 타순 있는 선수 중심
    hitters = lineup_df.copy()
    if "is_starting_pitcher" in hitters.columns:
        hitters = hitters[hitters["is_starting_pitcher"] != 1].copy()

    hitters = hitters.merge(
        season_hit_df,
        on=["player_id", "team_code"],
        how="left",
        suffixes=("", "_season")
    )

    # lineup 전체 평균
    def weighted_mean(x, value_col, weight_col="PA"):
        tmp = x[[value_col, weight_col]].dropna()
        if tmp.empty:
            return np.nan
        w = tmp[weight_col]
        v = tmp[value_col]
        if w.sum() == 0:
            return v.mean()
        return np.average(v, weights=w)

    team_rows = []

    for (game_id, team_code), g in hitters.groupby(["game_id", "team_code"]):
        row = {
            "game_id": game_id,
            "team_code": team_code,
            "lineup_n_hitters": len(g),
        }

        for stat in ["OPS", "OBP", "SLG"]:
            if stat in g.columns:
                row[f"lineup_{stat}"] = weighted_mean(g, stat, "PA")

        # top5
        if "batting_order" in g.columns:
            top5 = g[g["batting_order"].between(1, 5, inclusive="both")].copy()
        else:
            top5 = pd.DataFrame()

        row["top5_n"] = len(top5)

        for stat in ["OPS", "OBP"]:
            if not top5.empty and stat in top5.columns:
                row[f"top5_{stat}"] = weighted_mean(top5, stat, "PA")
            else:
                row[f"top5_{stat}"] = np.nan

        team_rows.append(row)

    return pd.DataFrame(team_rows)

In [ ]:
# 선발투수 feature 생성

def build_starting_pitcher_features(lineup_df: pd.DataFrame, season_pit_df: pd.DataFrame) -> pd.DataFrame:
    sp = lineup_df.copy()

    # 선발투수만
    if "is_starting_pitcher" in sp.columns:
        sp = sp[sp["is_starting_pitcher"] == 1].copy()
    else:
        sp = sp[sp["position"] == 1].copy()

    sp = sp.merge(
        season_pit_df,
        on=["player_id", "team_code"],
        how="left",
        suffixes=("", "_season")
    )

    keep_cols = [
        "game_id", "team_code", "player_id", "player_name",
        "ERA", "FIP", "WHIP", "K9", "BB9", "HR9", "KBB", "QS", "GS"
    ]
    keep_cols = [c for c in keep_cols if c in sp.columns]
    sp = sp[keep_cols].copy()

    rename_map = {
        "player_id": "sp_player_id",
        "player_name": "sp_player_name",
        "ERA": "sp_ERA",
        "FIP": "sp_FIP",
        "WHIP": "sp_WHIP",
        "K9": "sp_K9",
        "BB9": "sp_BB9",
        "HR9": "sp_HR9",
        "KBB": "sp_KBB",
        "QS": "sp_QS",
        "GS": "sp_GS",
    }
    sp = sp.rename(columns=rename_map)

    return sp

In [ ]:
def make_game_level_dataset(
    games_df_clean: pd.DataFrame,
    hitter_team_features: pd.DataFrame = None,
    sp_features: pd.DataFrame = None
) -> pd.DataFrame:
    """
    수정:
    - hitter_team_features가 None/empty여도 동작
    - 선발투수 feature만으로도 pred_df 생성 가능
    """
    df = games_df_clean.copy()

    # --------------------------------------------------
    # 1) hitter feature merge (있을 때만)
    # --------------------------------------------------
    if hitter_team_features is not None and not hitter_team_features.empty:
        home_hit = hitter_team_features.copy().rename(columns={
            "team_code": "home_team_code",
            "lineup_n_hitters": "home_lineup_n_hitters",
            "lineup_OPS": "home_lineup_OPS",
            "lineup_OBP": "home_lineup_OBP",
            "lineup_SLG": "home_lineup_SLG",
            "top5_n": "home_top5_n",
            "top5_OPS": "home_top5_OPS",
            "top5_OBP": "home_top5_OBP",
        })

        away_hit = hitter_team_features.copy().rename(columns={
            "team_code": "away_team_code",
            "lineup_n_hitters": "away_lineup_n_hitters",
            "lineup_OPS": "away_lineup_OPS",
            "lineup_OBP": "away_lineup_OBP",
            "lineup_SLG": "away_lineup_SLG",
            "top5_n": "away_top5_n",
            "top5_OPS": "away_top5_OPS",
            "top5_OBP": "away_top5_OBP",
        })

        df = df.merge(home_hit, on=["game_id", "home_team_code"], how="left")
        df = df.merge(away_hit, on=["game_id", "away_team_code"], how="left")

        # hitter diff 생성
        diff_pairs_hit = [
            ("lineup_OPS", "home_lineup_OPS", "away_lineup_OPS"),
            ("lineup_OBP", "home_lineup_OBP", "away_lineup_OBP"),
            ("lineup_SLG", "home_lineup_SLG", "away_lineup_SLG"),
            ("top5_OPS", "home_top5_OPS", "away_top5_OPS"),
            ("top5_OBP", "home_top5_OBP", "away_top5_OBP"),
        ]

        for new_name, h_col, a_col in diff_pairs_hit:
            if h_col in df.columns and a_col in df.columns:
                df[f"{new_name}_diff"] = df[h_col] - df[a_col]

    # --------------------------------------------------
    # 2) pitcher feature merge (필수)
    # --------------------------------------------------
    if sp_features is not None and not sp_features.empty:
        home_sp = sp_features.copy().rename(columns={
            "team_code": "home_team_code",
            "sp_player_id": "home_sp_player_id",
            "sp_player_name": "home_sp_player_name",
            "sp_ERA": "home_sp_ERA",
            "sp_FIP": "home_sp_FIP",
            "sp_WHIP": "home_sp_WHIP",
            "sp_K9": "home_sp_K9",
            "sp_BB9": "home_sp_BB9",
            "sp_HR9": "home_sp_HR9",
            "sp_KBB": "home_sp_KBB",
            "sp_QS": "home_sp_QS",
            "sp_GS": "home_sp_GS",
        })

        away_sp = sp_features.copy().rename(columns={
            "team_code": "away_team_code",
            "sp_player_id": "away_sp_player_id",
            "sp_player_name": "away_sp_player_name",
            "sp_ERA": "away_sp_ERA",
            "sp_FIP": "away_sp_FIP",
            "sp_WHIP": "away_sp_WHIP",
            "sp_K9": "away_sp_K9",
            "sp_BB9": "away_sp_BB9",
            "sp_HR9": "away_sp_HR9",
            "sp_KBB": "away_sp_KBB",
            "sp_QS": "away_sp_QS",
            "sp_GS": "away_sp_GS",
        })

        df = df.merge(home_sp, on=["game_id", "home_team_code"], how="left")
        df = df.merge(away_sp, on=["game_id", "away_team_code"], how="left")

        # pitcher diff 생성
        diff_pairs_pit = [
            ("sp_ERA", "home_sp_ERA", "away_sp_ERA"),
            ("sp_FIP", "home_sp_FIP", "away_sp_FIP"),
            ("sp_WHIP", "home_sp_WHIP", "away_sp_WHIP"),
            ("sp_K9", "home_sp_K9", "away_sp_K9"),
            ("sp_BB9", "home_sp_BB9", "away_sp_BB9"),
            ("sp_HR9", "home_sp_HR9", "away_sp_HR9"),
            ("sp_KBB", "home_sp_KBB", "away_sp_KBB"),
        ]

        for new_name, h_col, a_col in diff_pairs_pit:
            if h_col in df.columns and a_col in df.columns:
                df[f"{new_name}_diff"] = df[h_col] - df[a_col]

    return df

NameError: name 'pd' is not defined

In [ ]:
model_df

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,top5_OPS_diff,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,y_home_win
0,20230041,2023-04-11 09:30:00,2023,6002,10001,1001,14770,11379,최승용,최원태,...,-0.013535,-0.012362,0.72,0.24084,0.19,-0.211,0.207,0.114,-0.2779,1
1,20230041,2023-04-11 09:30:00,2023,6002,10001,1001,14770,11379,최승용,최원태,...,-0.013535,-0.012362,0.72,0.24084,0.19,-1.471,-0.694,-0.285,0.0589,1
2,20230042,2023-04-11 09:30:00,2023,1001,9002,7003,14113,14581,원태인,오원석,...,-0.000441,0.004849,-1.99,-0.73886,-0.30,0.645,-2.253,0.216,1.7246,0
3,20230043,2023-04-11 09:30:00,2023,2002,7002,6001,10058,14619,양현종,남지민,...,0.036835,0.011606,-2.87,-0.10633,-0.44,1.504,-0.580,0.684,1.0016,0
4,20230044,2023-04-11 09:30:00,2023,3001,5002,8001,15011,15486,반즈,박명근,...,-0.105756,-0.048248,-1.80,-1.41826,-0.17,0.754,-1.950,-0.384,1.1964,1
5,20230045,2023-04-11 09:30:00,2023,11001,12001,8005,13085,15542,신민혁,슐서,...,0.052905,0.036511,-1.64,-0.32167,-0.41,0.814,-0.693,-0.054,1.3800,1
6,20230091,2023-04-22 08:00:00,2023,6002,12001,1001,14770,15542,최승용,슐서,...,-0.010307,0.003771,-1.65,-0.30797,-0.26,0.307,0.220,-0.357,-0.0882,1
7,20230092,2023-04-22 08:00:00,2023,9002,10001,2001,15528,11379,맥카티,최원태,...,0.031180,-0.001007,0.14,-0.15690,0.14,1.171,0.565,0.076,-0.1119,1
8,20230092,2023-04-22 08:00:00,2023,9002,10001,2001,15528,11379,맥카티,최원태,...,0.031180,-0.001007,0.14,-0.15690,0.14,-0.089,-0.336,-0.323,0.2249,1
9,20230093,2023-04-22 08:00:00,2023,7002,5002,4001,14619,10652,남지민,임찬규,...,-0.046549,-0.042822,3.03,-0.35205,0.43,-0.912,-0.253,-0.622,-0.1382,0


In [ ]:
'''# 실행하여 경기단위 df 만들기 

# 1. 전처리
games_model = prepare_games_for_modeling(games_df_clean if "games_df_clean" in globals() else games_df)
lineup_model = prepare_lineup(lineup_df)
season_hit_model = prepare_season_hitters(season_hit_df)
season_pit_model = prepare_season_pitchers(season_pit_df)

# 2. 팀 타자 feature
hitter_team_features = build_hitter_team_features(lineup_model, season_hit_model)

# 3. 선발투수 feature
sp_features = build_starting_pitcher_features(lineup_model, season_pit_model)

# 4. 최종 경기 단위 데이터셋
model_df = make_game_level_dataset(
    games_df_clean=games_model,
    hitter_team_features=hitter_team_features,
    sp_features=sp_features
)

display(model_df.head())
print(model_df.shape)'''

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,top5_OPS_diff,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,y_home_win
0,20230041,2023-04-11 09:30:00,2023,6002,10001,1001,14770,11379,최승용,최원태,...,-0.013535,-0.012362,0.72,0.24084,0.19,-0.211,0.207,0.114,-0.2779,1
1,20230041,2023-04-11 09:30:00,2023,6002,10001,1001,14770,11379,최승용,최원태,...,-0.013535,-0.012362,0.72,0.24084,0.19,-1.471,-0.694,-0.285,0.0589,1
2,20230042,2023-04-11 09:30:00,2023,1001,9002,7003,14113,14581,원태인,오원석,...,-0.000441,0.004849,-1.99,-0.73886,-0.30,0.645,-2.253,0.216,1.7246,0
3,20230043,2023-04-11 09:30:00,2023,2002,7002,6001,10058,14619,양현종,남지민,...,0.036835,0.011606,-2.87,-0.10633,-0.44,1.504,-0.580,0.684,1.0016,0
4,20230044,2023-04-11 09:30:00,2023,3001,5002,8001,15011,15486,반즈,박명근,...,-0.105756,-0.048248,-1.80,-1.41826,-0.17,0.754,-1.950,-0.384,1.1964,1


(17, 70)


In [ ]:
problem_pitchers = [12930,
 10126,
 12944,
 10131,
 16148,
 11284,
 14613,
 14617,
 15001,
 16153,
 10652,
 15643,
 15646,
 16160,
 14113,
 15011,
 13091,
 15143,
 15528,
 15146,
 15531,
 15147,
 15533,
 11310,
 11311,
 15536,
 15153,
 14770,
 15535,
 14769,
 14132,
 14128,
 12852,
 11318,
 16065,
 15045,
 15432,
 10058,
 15485,
 14808,
 16088,
 16123,
 11232,
 10609,
 11379,
 13941,
 13942,
 14581,
 10363,
 14588,
 14589]

summary_rows = []

for pid in problem_pitchers:
    res = api_get("prediction/playerSeason", {"p_no": str(pid)})
    time.sleep(0.2)
    

    basic_df = pd.DataFrame(res.get("basic", {}).get("list", []))
    deepen_df = pd.DataFrame(res.get("deepen", {}).get("list", []))

    if not basic_df.empty and "year" in basic_df.columns:
        basic_df["year"] = pd.to_numeric(basic_df["year"], errors="coerce")

    if not deepen_df.empty and "year" in deepen_df.columns:
        deepen_df["year"] = pd.to_numeric(deepen_df["year"], errors="coerce")

    basic_n_2023 = ((basic_df["year"] == 2023).sum() if "year" in basic_df.columns else 0)
    deepen_n_2023 = ((deepen_df["year"] == 2023).sum() if "year" in deepen_df.columns else 0)

    summary_rows.append({
        "player_id": pid,
        "basic_n_2023": int(basic_n_2023),
        "deepen_n_2023": int(deepen_n_2023),
    })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

KeyboardInterrupt: 

In [ ]:
games_model

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_score,home_win,game_state,game_time,weather,temperature,humidity,windDirection,windSpeed,rainprobability
0,20230041,2023-04-11 09:30:00,2023,6002,10001,1001,14770,11379,최승용,최원태,...,4,1,3,18:30:00,10400,13.0,84,21100,3.1,None
1,20230042,2023-04-11 09:30:00,2023,1001,9002,7003,14113,14581,원태인,오원석,...,5,0,3,18:30:00,10400,20.2,59,20900,1.8,None
2,20230043,2023-04-11 09:30:00,2023,2002,7002,6001,10058,14619,양현종,남지민,...,5,0,3,18:30:00,10400,16.9,92,21300,2.2,None
3,20230044,2023-04-11 09:30:00,2023,3001,5002,8001,15011,15486,반즈,박명근,...,5,1,3,18:30:00,10500,18.4,70,20900,3.4,None
4,20230045,2023-04-11 09:30:00,2023,11001,12001,8005,13085,15542,신민혁,슐서,...,0,1,3,18:30:00,10400,18.3,76,21500,1.0,None
5,20230091,2023-04-22 08:00:00,2023,6002,12001,1001,14770,15542,최승용,슐서,...,1,1,3,17:00:00,10100,21.7,13,20500,2.0,None
6,20230092,2023-04-22 08:00:00,2023,9002,10001,2001,15528,11379,맥카티,최원태,...,2,1,3,17:00:00,10400,21.2,11,20100,2.5,None
7,20230093,2023-04-22 08:00:00,2023,7002,5002,4001,14619,10652,남지민,임찬규,...,3,0,3,17:00:00,10400,20.0,21,20700,4.6,None
8,20230094,2023-04-22 08:00:00,2023,2002,1001,6001,10058,14113,양현종,원태인,...,2,1,3,17:00:00,10100,21.3,34,21500,2.6,None
9,20230095,2023-04-22 08:00:00,2023,11001,3001,8005,13085,15011,신민혁,반즈,...,10,0,3,17:00:00,10100,17.3,27,20300,2.4,None


In [ ]:
season_pit_df

,player_id,year,team_code,GS,IP,IP_decimal,ERA,FIP,WHIP,K9,BB9,HR9,KBB,QS,snapshot_date
0,10363,2023,1001,18,100.2,100.666667,3.67,3.90591,1.29,5.454,2.682,0.536,2.0333,8,2023-04-22
4,11404,2023,1001,3,19.1,19.333333,7.91,5.90273,2.07,5.121,5.586,0.931,0.9167,0,2023-04-22
6,12530,2023,1001,0,9.1,9.333333,4.82,8.85286,1.82,7.714,6.750,2.893,1.1429,0,2023-04-22
8,10685,2023,1001,5,64.0,64.000000,4.50,4.39928,1.38,6.891,3.094,0.984,2.2273,0,2023-04-22
11,10523,2023,1001,0,43.0,43.000000,4.81,3.62835,1.40,5.860,1.047,0.628,5.6000,0,2023-04-22
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
385,13130,2023,12001,0,24.2,24.666667,6.93,7.44118,1.42,2.919,3.284,2.189,0.8889,0,2023-04-22
388,15143,2023,12001,29,160.0,160.000000,3.54,3.14428,1.21,8.831,2.531,0.675,3.4889,11,2023-04-22
390,11284,2023,12001,4,35.0,35.000000,6.69,4.36357,1.91,4.371,1.543,0.771,2.8333,0,2023-04-22
393,14108,2023,12001,0,73.2,73.666667,3.42,4.22530,1.19,4.887,2.443,0.611,2.0000,0,2023-04-22


In [ ]:
season_hit_model['team_code'].unique()

array(['1001', '10001', '2002', '3001', '5002', '6002', '7002', '9002',
       '11001', '12001'], dtype=object)

In [ ]:
hitter_team_features

,game_id,team_code,lineup_n_hitters,lineup_OPS,lineup_OBP,lineup_SLG,top5_n,top5_OPS,top5_OBP
0,20230041,10001,9,0.741863,0.354533,0.387331,5,0.771619,0.367792
1,20230041,6002,9,0.747009,0.345395,0.401614,5,0.758084,0.355430
2,20230042,1001,9,0.767009,0.349069,0.417940,5,0.798128,0.361185
3,20230042,9002,9,0.760653,0.348073,0.412580,5,0.798569,0.356336
4,20230043,2002,9,0.735926,0.345530,0.390396,5,0.779967,0.358466
5,20230043,7002,9,0.704245,0.329925,0.374320,5,0.743132,0.346860
6,20230044,3001,9,0.695640,0.333647,0.361993,5,0.713518,0.343895
7,20230044,5002,9,0.783629,0.371367,0.412263,5,0.819274,0.392143
8,20230045,11001,9,0.773582,0.366171,0.407411,5,0.833821,0.390592
9,20230045,12001,9,0.752064,0.350972,0.401092,5,0.780916,0.354081


In [ ]:
sp_features.head(10)

,game_id,team_code,sp_player_id,sp_player_name,sp_ERA,sp_FIP,sp_WHIP,sp_K9,sp_BB9,sp_HR9,sp_KBB,sp_QS,sp_GS
0,20230041,6002,14770,최승용,3.97,4.02856,1.35,6.649,2.757,0.730,2.4118,4.0,20.0
1,20230041,10001,11379,최원태,3.25,3.78772,1.16,6.860,2.550,0.616,2.6897,11.0,17.0
2,20230041,10001,11379,최원태,3.25,3.78772,1.16,8.120,3.451,1.015,2.3529,11.0,17.0
3,20230042,1001,14113,원태인,3.24,4.16595,1.27,6.120,2.040,0.900,3.0000,17.0,26.0
4,20230042,9002,14581,오원석,5.23,4.90481,1.57,5.475,4.293,0.684,1.2754,7.0,27.0
5,20230043,2002,10058,양현종,3.58,3.69782,1.34,7.000,2.526,0.684,2.7708,14.0,29.0
6,20230043,7002,14619,남지민,6.45,3.80415,1.78,5.496,3.106,0.000,1.7692,1.0,7.0
7,20230044,3001,15011,반즈,3.28,3.40310,1.33,7.767,2.959,0.317,2.6250,18.0,30.0
8,20230044,5002,15486,박명근,5.08,4.82136,1.50,7.013,4.909,0.701,1.4286,0.0,1.0
9,20230045,11001,13085,신민혁,3.98,4.01486,1.20,7.156,1.844,1.033,3.8800,5.0,24.0


In [ ]:
def _get_last_team_code(group: pd.DataFrame):
    """
    같은 player_id, year 그룹 내에서 마지막 team_code 반환
    - API에서 들어온 행 순서를 그대로 믿고 마지막 팀으로 간주
    """
    if "team_code" not in group.columns:
        return np.nan

    vals = group["team_code"].dropna().astype(str)
    if len(vals) == 0:
        return np.nan

    return vals.iloc[-1]

In [ ]:
def deduplicate_hitter_season_last_team(df: pd.DataFrame) -> pd.DataFrame:
    """
    같은 선수의 같은 시즌 여러 행(팀 이동 등)을 1행으로 통합
    - team_code: 마지막 팀으로 통일
    - counting stat: 합산
    - rate stat: PA 가중평균 (불가능하면 평균)
    """
    df = df.copy()

    # year 숫자화
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    # 숫자 컬럼 변환
    numeric_cols = [
        "PA", "AB", "H", "HR", "RBI", "BB", "SO",
        "AVG", "OBP", "SLG", "OPS", "wOBA", "wRCplus", "BBK"
    ]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    out_rows = []

    for (player_id, year), g in df.groupby(["player_id", "year"], sort=False):
        row = {
            "player_id": str(player_id),
            "year": year,
            # 마지막 팀으로 통일
            "team_code": _get_last_team_code(g),
        }

        # 이름은 마지막 값 사용
        if "player_name" in g.columns:
            name_vals = g["player_name"].dropna()
            row["player_name"] = name_vals.iloc[-1] if len(name_vals) > 0 else np.nan

        # -------------------------
        # counting stat 합산
        # -------------------------
        count_cols = ["PA", "AB", "H", "HR", "RBI", "BB", "SO"]
        for c in count_cols:
            if c in g.columns:
                row[c] = g[c].sum(min_count=1)

        # -------------------------
        # 재계산/가중평균용 기반 값
        # -------------------------
        PA = row.get("PA", np.nan)
        AB = row.get("AB", np.nan)
        H  = row.get("H", np.nan)

        # AVG = H / AB
        if "AVG" in df.columns:
            row["AVG"] = (H / AB) if pd.notna(H) and pd.notna(AB) and AB != 0 else np.nan

        # 단순 재계산이 어려운 rate stat
        weighted_rate_cols = ["OBP", "SLG", "OPS", "wOBA", "wRCplus", "BBK"]

        for c in weighted_rate_cols:
            if c in g.columns:
                tmp = g[[c]].copy()

                # PA가 있으면 PA 가중평균
                if "PA" in g.columns:
                    tmp["PA"] = pd.to_numeric(g["PA"], errors="coerce")
                    tmp = tmp.dropna(subset=[c])

                    if not tmp.empty and tmp["PA"].fillna(0).sum() > 0:
                        row[c] = np.average(tmp[c], weights=tmp["PA"].fillna(0))
                    else:
                        row[c] = tmp[c].mean() if not tmp.empty else np.nan
                else:
                    row[c] = pd.to_numeric(g[c], errors="coerce").mean()

        out_rows.append(row)

    out = pd.DataFrame(out_rows)

    ordered_cols = [
        "player_id", "player_name", "team_code", "year",
        "PA", "AB", "H", "HR", "RBI", "BB", "SO",
        "AVG", "OBP", "SLG", "OPS", "wOBA", "wRCplus", "BBK"
    ]
    ordered_cols = [c for c in ordered_cols if c in out.columns]

    return out[ordered_cols].copy()

In [ ]:
def deduplicate_pitcher_season_last_team(df: pd.DataFrame) -> pd.DataFrame:
    """
    같은 선수의 같은 시즌 여러 행(팀 이동 등)을 1행으로 통합
    - team_code: 마지막 팀으로 통일
    - counting stat: 합산
    - rate stat: IP 가중평균
    """
    df = df.copy()

    # year 숫자화
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    # 숫자 컬럼 변환
    numeric_cols = [
        "GS", "IP", "IP_decimal", "ERA", "FIP", "WHIP",
        "K9", "BB9", "HR9", "KBB", "QS"
    ]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    out_rows = []

    for (player_id, year), g in df.groupby(["player_id", "year"], sort=False):
        row = {
            "player_id": str(player_id),
            "year": year,
            # 마지막 팀으로 통일
            "team_code": _get_last_team_code(g),
        }

        # 이름은 마지막 값 사용
        if "player_name" in g.columns:
            name_vals = g["player_name"].dropna()
            row["player_name"] = name_vals.iloc[-1] if len(name_vals) > 0 else np.nan

        # -------------------------
        # counting stat 합산
        # -------------------------
        count_cols = ["GS", "QS"]
        for c in count_cols:
            if c in g.columns:
                row[c] = g[c].sum(min_count=1)

        # -------------------------
        # IP는 decimal 기준으로 합산
        # -------------------------
        if "IP_decimal" in g.columns:
            row["IP_decimal"] = g["IP_decimal"].sum(min_count=1)
        elif "IP" in g.columns:
            row["IP"] = g["IP"].sum(min_count=1)

        # -------------------------
        # rate stat은 IP 가중평균
        # -------------------------
        weight_col = "IP_decimal" if "IP_decimal" in g.columns else ("IP" if "IP" in g.columns else None)
        weighted_rate_cols = ["ERA", "FIP", "WHIP", "K9", "BB9", "HR9", "KBB"]

        for c in weighted_rate_cols:
            if c in g.columns:
                tmp = g[[c]].copy()

                if weight_col is not None:
                    tmp[weight_col] = pd.to_numeric(g[weight_col], errors="coerce")
                    tmp = tmp.dropna(subset=[c])

                    if not tmp.empty and tmp[weight_col].fillna(0).sum() > 0:
                        row[c] = np.average(tmp[c], weights=tmp[weight_col].fillna(0))
                    else:
                        row[c] = tmp[c].mean() if not tmp.empty else np.nan
                else:
                    row[c] = pd.to_numeric(g[c], errors="coerce").mean()

        out_rows.append(row)

    out = pd.DataFrame(out_rows)

    ordered_cols = [
        "player_id", "player_name", "team_code", "year",
        "GS", "IP_decimal", "ERA", "FIP", "WHIP",
        "K9", "BB9", "HR9", "KBB", "QS"
    ]
    ordered_cols = [c for c in ordered_cols if c in out.columns]

    return out[ordered_cols].copy()

In [ ]:
def deduplicate_pitcher_season_last_team(df: pd.DataFrame) -> pd.DataFrame:
    """
    같은 선수의 같은 시즌 여러 행(팀 이동 등)을 1행으로 통합
    - team_code: 마지막 팀으로 통일
    - counting stat: 합산
    - rate stat: IP 가중평균
    """
    df = df.copy()

    # year 숫자화
    df["year"] = pd.to_numeric(df["year"], errors="coerce")

    # 숫자 컬럼 변환
    numeric_cols = [
        "GS", "IP", "IP_decimal", "ERA", "FIP", "WHIP",
        "K9", "BB9", "HR9", "KBB", "QS"
    ]
    for c in numeric_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    out_rows = []

    for (player_id, year), g in df.groupby(["player_id", "year"], sort=False):
        row = {
            "player_id": str(player_id),
            "year": year,
            # 마지막 팀으로 통일
            "team_code": _get_last_team_code(g),
        }

        # 이름은 마지막 값 사용
        if "player_name" in g.columns:
            name_vals = g["player_name"].dropna()
            row["player_name"] = name_vals.iloc[-1] if len(name_vals) > 0 else np.nan

        # -------------------------
        # counting stat 합산
        # -------------------------
        count_cols = ["GS", "QS"]
        for c in count_cols:
            if c in g.columns:
                row[c] = g[c].sum(min_count=1)

        # -------------------------
        # IP는 decimal 기준으로 합산
        # -------------------------
        if "IP_decimal" in g.columns:
            row["IP_decimal"] = g["IP_decimal"].sum(min_count=1)
        elif "IP" in g.columns:
            row["IP"] = g["IP"].sum(min_count=1)

        # -------------------------
        # rate stat은 IP 가중평균
        # -------------------------
        weight_col = "IP_decimal" if "IP_decimal" in g.columns else ("IP" if "IP" in g.columns else None)
        weighted_rate_cols = ["ERA", "FIP", "WHIP", "K9", "BB9", "HR9", "KBB"]

        for c in weighted_rate_cols:
            if c in g.columns:
                tmp = g[[c]].copy()

                if weight_col is not None:
                    tmp[weight_col] = pd.to_numeric(g[weight_col], errors="coerce")
                    tmp = tmp.dropna(subset=[c])

                    if not tmp.empty and tmp[weight_col].fillna(0).sum() > 0:
                        row[c] = np.average(tmp[c], weights=tmp[weight_col].fillna(0))
                    else:
                        row[c] = tmp[c].mean() if not tmp.empty else np.nan
                else:
                    row[c] = pd.to_numeric(g[c], errors="coerce").mean()

        out_rows.append(row)

    out = pd.DataFrame(out_rows)

    ordered_cols = [
        "player_id", "player_name", "team_code", "year",
        "GS", "IP_decimal", "ERA", "FIP", "WHIP",
        "K9", "BB9", "HR9", "KBB", "QS"
    ]
    ordered_cols = [c for c in ordered_cols if c in out.columns]

    return out[ordered_cols].copy()

In [ ]:
def prepare_season_pitchers(season_pit_df: pd.DataFrame) -> pd.DataFrame:
    df = season_pit_df.copy()

    df["player_id"] = df["player_id"].astype(str)

    if "team_code" in df.columns:
        df["team_code"] = df["team_code"].astype(str)

    # year -> season 맞춤
    if "year" in df.columns:
        df["year"] = pd.to_numeric(df["year"], errors="coerce")
        df["season"] = df["year"]

    num_cols = ["GS", "IP", "IP_decimal", "ERA", "FIP", "WHIP", "K9", "BB9", "HR9", "KBB", "QS"]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

In [ ]:
def build_modeling_data_pipeline(
    date_list,
    team_codes=None,
    roster_code="1",
    sleep_sec=0.2,
    verbose=True
):
    """
    date_list를 입력받아
    1) games 수집
    2) 정상 경기만 필터
    3) lineup 수집
    4) roster 수집 (date_list 전체 기준)
    5) player season 수집
    6) 시즌 중 팀 이동 선수 처리 (마지막 팀으로 통일 + 시즌 성적 통합)
    7) pitcher 결측/무한대 처리
    8) 경기 단위 model_df 생성
    까지를 한 번에 수행

    Parameters
    ----------
    date_list : list[str]
        예: ["2025-04-22", "2025-04-23", "2025-04-24"]

    team_codes : list[str], optional
        팀코드 리스트. None이면 KBO 10개 구단 기본값 사용.

    roster_code : str, default "1"
        playerRoster 호출용 code 파라미터

    sleep_sec : float, default 0.2
        반복 API 호출 간 텀 (429 방지)

    verbose : bool, default True
        진행 로그 출력 여부

    Returns
    -------
    result : dict
        {
            "games_df": ...,
            "games_df_clean": ...,
            "lineup_df": ...,
            "roster_df": ...,
            "season_hit_df": ...,
            "season_pit_df": ...,
            "games_model": ...,
            "lineup_model": ...,
            "season_hit_model": ...,
            "season_pit_model": ...,
            "hitter_team_features": ...,
            "sp_features": ...,
            "model_df": ...
        }
    """

    # --------------------------------------------------
    # 0) 기본 팀코드 설정
    # --------------------------------------------------
    if team_codes is None:
        team_codes = ["1001", "2002", "3001", "5002", "6002", "7002", "9002", "10001", "11001", "12001"]
        # 삼성, KIA, 롯데, LG, 두산, 한화, SSG, 키움, NC, KT

    # --------------------------------------------------
    # 1) games 수집
    # --------------------------------------------------
    if verbose:
        print("=== 1. collect_games ===")
    games_df = collect_games(date_list, sleep_sec=sleep_sec)

    # --------------------------------------------------
    # 2) 정상 경기만 필터
    # game_state == 3 이고 점수가 존재하는 경기만 사용
    # --------------------------------------------------
    if verbose:
        print("=== 2. filter normal games ===")

    games_df_clean = games_df[
        (games_df["game_state"] == 3) &
        (games_df["home_score"].notna()) &
        (games_df["away_score"].notna())
    ].copy()

    if verbose:
        print(f"games_df.shape       = {games_df.shape}")
        print(f"games_df_clean.shape = {games_df_clean.shape}")

    # --------------------------------------------------
    # 3) lineup 수집
    # 정상 경기만 대상으로 호출
    # --------------------------------------------------
    if verbose:
        print("=== 3. collect_lineups ===")

    lineup_df = collect_lineups(games_df_clean, sleep_sec=sleep_sec)

    if verbose:
        print(f"lineup_df.shape = {lineup_df.shape}")

    # --------------------------------------------------
    # 4) roster 수집
    # 특정 하루만 수집하면 선수 누락 가능하므로
    # date_list 전체 기준으로 roster를 합침
    # --------------------------------------------------
    if verbose:
        print("=== 4. collect_rosters (all dates) ===")

    roster_dfs = []

    for dt in date_list:
        if verbose:
            print(f"roster date: {dt}")

        tmp_roster = collect_rosters(
            date=dt,
            team_codes=team_codes,
            code=roster_code,
            sleep_sec=sleep_sec
        )

        if not tmp_roster.empty:
            roster_dfs.append(tmp_roster)

    if roster_dfs:
        roster_df = pd.concat(roster_dfs, ignore_index=True).drop_duplicates()
    else:
        roster_df = pd.DataFrame(columns=["snapshot_date", "team_code", "player_id", "player_name", "join_date"])

    if verbose:
        print(f"roster_df.shape = {roster_df.shape}")

    # --------------------------------------------------
    # 5) player_ids 추출
    # --------------------------------------------------
    if roster_df.empty:
        raise ValueError("roster_df가 비어 있습니다. playerSeason 수집을 진행할 수 없습니다.")

    player_ids = roster_df["player_id"].astype(str).unique().tolist()

    if verbose:
        print(f"n_unique_player_ids = {len(player_ids)}")

    # --------------------------------------------------
    # 6) playerSeason 수집
    # --------------------------------------------------
    snapshot_date = date_list[0]

    if verbose:
        print("=== 5. collect_player_season ===")

    season_hit_df, season_pit_df = collect_player_season(
        player_ids,
        snapshot_date=snapshot_date,
        sleep_sec=sleep_sec
    )

    # --------------------------------------------------
    # 7) target season 필터
    # 현재 pipeline은 date_list가 한 시즌이라고 가정
    # --------------------------------------------------
    target_year = pd.to_datetime(date_list[0]).year

    season_hit_df["year"] = pd.to_numeric(season_hit_df["year"], errors="coerce")
    season_pit_df["year"] = pd.to_numeric(season_pit_df["year"], errors="coerce")

    season_hit_df = season_hit_df[season_hit_df["year"] == target_year].copy()
    season_pit_df = season_pit_df[season_pit_df["year"] == target_year].copy()

    # --------------------------------------------------
    # 8) 시즌 중 팀 이동 선수 처리
    # 핵심:
    # - player_id + year 기준 1행으로 압축
    # - team_code는 마지막 팀으로 통일
    # - season 성적은 통합
    # --------------------------------------------------
    if verbose:
        print("=== 6. deduplicate season stats (last team + aggregate season) ===")

    season_hit_df = deduplicate_hitter_season_last_team(season_hit_df)
    season_pit_df = deduplicate_pitcher_season_last_team(season_pit_df)

    if verbose:
        print(f"season_hit_df.shape after dedup = {season_hit_df.shape}")
        print(f"season_pit_df.shape after dedup = {season_pit_df.shape}")

    # --------------------------------------------------
    # 9) pitcher 특수값 처리
    # ERA/FIP/WHIP/KBB의 inf, 결측 등을 99.99로 치환
    # --------------------------------------------------
    if verbose:
        print("=== 7. clean pitcher season stats ===")

    pit_fill_cols = [c for c in ["ERA", "FIP", "WHIP", "KBB"] if c in season_pit_df.columns]
    if pit_fill_cols:
        season_pit_df[pit_fill_cols] = (
            season_pit_df[pit_fill_cols]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(99.99)
        )

    # --------------------------------------------------
    # 10) 모델링용 전처리
    # --------------------------------------------------
    if verbose:
        print("=== 8. prepare modeling tables ===")

    games_model = prepare_games_for_modeling(games_df_clean)
    lineup_model = prepare_lineup(lineup_df)
    season_hit_model = prepare_season_hitters(season_hit_df)
    season_pit_model = prepare_season_pitchers(season_pit_df)

    # --------------------------------------------------
    # 11) lineup에 season 붙이기
    # season stats를 player_id + season 기준으로 merge할 수 있게 준비
    # --------------------------------------------------
    lineup_model = lineup_model.merge(
        games_model[["game_id", "season"]],
        on="game_id",
        how="left"
    )

    # --------------------------------------------------
    # 12) 팀 타자 feature 생성
    # 주의:
    # build_hitter_team_features 내부에서
    # season_hit_model과 merge할 때 on=["player_id", "season"] 기준이 되도록
    # 함수도 같이 수정해서 쓰는 것을 권장
    # --------------------------------------------------
    if verbose:
        print("=== 9. build hitter team features ===")

    hitter_team_features = build_hitter_team_features(lineup_model, season_hit_model)

    if verbose:
        print(f"hitter_team_features.shape = {hitter_team_features.shape}")

    # --------------------------------------------------
    # 13) 선발투수 feature 생성
    # 주의:
    # build_starting_pitcher_features 내부에서
    # season_pit_model과 merge할 때 on=["player_id", "season"] 기준이 되도록
    # 함수도 같이 수정해서 쓰는 것을 권장
    # --------------------------------------------------
    if verbose:
        print("=== 10. build starting pitcher features ===")

    sp_features = build_starting_pitcher_features(lineup_model, season_pit_model)

    if verbose:
        print(f"sp_features.shape = {sp_features.shape}")

        # 경기-팀당 1행인지 확인
        if not sp_features.empty:
            print("sp_features rows per (game_id, team_code):")
            print(sp_features.groupby(["game_id", "team_code"]).size().value_counts().head())

    # --------------------------------------------------
    # 14) 최종 경기 단위 데이터셋 생성
    # --------------------------------------------------
    if verbose:
        print("=== 11. make game-level dataset ===")

    model_df = make_game_level_dataset(
        games_df_clean=games_model,
        hitter_team_features=hitter_team_features,
        sp_features=sp_features
    )

    if verbose:
        print(f"model_df.shape = {model_df.shape}")

        # 같은 경기 중복 여부 확인
        print("game_id duplication check:")
        print(model_df["game_id"].value_counts().value_counts().head())

    # --------------------------------------------------
    # 15) diff feature 결측 확인
    # --------------------------------------------------
    if verbose:
        diff_cols = [c for c in model_df.columns if c.endswith("_diff")]
        if diff_cols:
            na_summary = model_df[diff_cols].isna().sum().sort_values(ascending=False)
            print("Top NA counts in diff features:")
            print(na_summary.head(10))

    return {
        "games_df": games_df,
        "games_df_clean": games_df_clean,
        "lineup_df": lineup_df,
        "roster_df": roster_df,
        "season_hit_df": season_hit_df,
        "season_pit_df": season_pit_df,
        "games_model": games_model,
        "lineup_model": lineup_model,
        "season_hit_model": season_hit_model,
        "season_pit_model": season_pit_model,
        "hitter_team_features": hitter_team_features,
        "sp_features": sp_features,
        "model_df": model_df,
    }

In [ ]:
result = build_modeling_data_pipeline(
    date_list=["2025-04-22", "2025-04-23", "2025-04-24"],
    sleep_sec=0.2,
    verbose=True
)

model_df = result["model_df"]
display(model_df.head())
print(model_df.shape)

=== 1. collect_games ===
[games] 2025-04-22: 5 rows
[games] 2025-04-23: 5 rows
[games] 2025-04-24: 5 rows


C:\Users\User\AppData\Local\Temp\ipykernel_16088\1047996336.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(all_dfs, ignore_index=True).drop_duplicates()


=== 2. filter normal games ===
games_df.shape       = (15, 21)
games_df_clean.shape = (13, 21)
=== 3. collect_lineups ===
[lineup] 20250131: 20 rows
[lineup] 20250134: 20 rows
[lineup] 20250135: 20 rows
[lineup] 20250136: 20 rows
[lineup] 20250137: 20 rows
[lineup] 20250138: 20 rows
[lineup] 20250139: 20 rows
[lineup] 20250140: 20 rows
[lineup] 20250141: 20 rows
[lineup] 20250142: 20 rows
[lineup] 20250143: 20 rows
[lineup] 20250144: 20 rows
[lineup] 20250145: 20 rows
lineup_df.shape = (260, 13)
=== 4. collect_rosters (all dates) ===
roster date: 2025-04-22
[roster] 2025-04-22 team=1001: 28 rows
[roster] 2025-04-22 team=2002: 28 rows
[roster] 2025-04-22 team=3001: 28 rows
[roster] 2025-04-22 team=5002: 28 rows
[roster] 2025-04-22 team=6002: 28 rows
[roster] 2025-04-22 team=7002: 28 rows
[roster] 2025-04-22 team=9002: 28 rows
[roster] 2025-04-22 team=10001: 28 rows
[roster] 2025-04-22 team=11001: 28 rows
[roster] 2025-04-22 team=12001: 28 rows
roster date: 2025-04-23
[roster] 2025-04-23

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,top5_OPS_diff,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,y_home_win
0,20250131,2025-04-22 09:30:00,2025,5002,11001,1001,10652,13085,임찬규,신민혁,...,0.043075,0.002417,-1.74,-1.36062,-0.05,0.279,0.472,-1.063,-0.5558,0
1,20250134,2025-04-22 09:30:00,2025,12001,9002,2002,14581,10126,오원석,김광현,...,-0.012903,-0.008249,-1.33,0.26239,-0.11,-0.940,0.412,-0.132,-0.5869,1
2,20250135,2025-04-22 09:30:00,2025,10001,6002,1003,11222,14770,하영민,최승용,...,-0.022400,-0.015781,0.58,-0.58024,0.02,2.372,-0.378,0.144,1.2961,1
3,20250136,2025-04-23 09:30:00,2025,5002,11001,1001,14846,16350,송승기,로건,...,0.023721,-0.018078,-1.03,-0.29169,-0.05,0.061,-0.424,0.002,0.3271,1
4,20250137,2025-04-23 09:30:00,2025,3001,7002,8001,15011,16153,반즈,와이스,...,-0.050807,0.008403,2.45,1.08821,0.38,-2.938,0.529,0.133,-1.4611,0


(13, 70)


In [ ]:
model_df["game_id"].value_counts().value_counts()

count
1    13
Name: count, dtype: int64

In [ ]:
from datetime import datetime, timedelta

def generate_date_list(start_date: str, end_date: str):
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end   = datetime.strptime(end_date, "%Y-%m-%d")

    date_list = []
    cur = start

    while cur <= end:
        date_list.append(cur.strftime("%Y-%m-%d"))
        cur += timedelta(days=1)

    return date_list

In [ ]:
date_list_23 = generate_date_list("2023-03-22", "2023-10-01")

In [ ]:
date_list_24 = generate_date_list("2024-03-22", "2024-10-01")

In [ ]:
date_list_25 = generate_date_list("2025-03-22", "2025-10-01")

In [ ]:
result_23 = build_modeling_data_pipeline(
    date_list=date_list_23,
    sleep_sec=0.2,
    verbose=True
)

model_df_23 = result_23["model_df"]

display(model_df_23.head())
print(model_df_23.shape)

=== 1. collect_games ===
[games] 2023-03-22: 0 rows
[games] 2023-03-23: 5 rows
[games] 2023-03-24: 5 rows
[games] 2023-03-25: 5 rows
[games] 2023-03-26: 5 rows
[games] 2023-03-27: 5 rows
[games] 2023-03-28: 5 rows
[games] 2023-03-29: 0 rows
[games] 2023-03-30: 0 rows
[games] 2023-03-31: 0 rows
[games] 2023-04-01: 5 rows
[games] 2023-04-02: 5 rows
[games] 2023-04-03: 0 rows
[games] 2023-04-04: 5 rows
[games] 2023-04-05: 5 rows
[games] 2023-04-06: 5 rows
[games] 2023-04-07: 5 rows
[games] 2023-04-08: 5 rows
[games] 2023-04-09: 5 rows
[games] 2023-04-10: 0 rows
[games] 2023-04-11: 5 rows
[games] 2023-04-12: 5 rows
[games] 2023-04-13: 5 rows
[games] 2023-04-14: 5 rows
[games] 2023-04-15: 5 rows
[games] 2023-04-16: 5 rows
[games] 2023-04-17: 0 rows
[games] 2023-04-18: 5 rows
[games] 2023-04-19: 5 rows
[games] 2023-04-20: 5 rows
[games] 2023-04-21: 5 rows
[games] 2023-04-22: 5 rows
[games] 2023-04-23: 5 rows
[games] 2023-04-24: 0 rows
[games] 2023-04-25: 5 rows
[games] 2023-04-26: 5 rows
[ga

C:\Users\User\AppData\Local\Temp\ipykernel_27996\1047996336.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(all_dfs, ignore_index=True).drop_duplicates()


=== 2. filter normal games ===
games_df.shape       = (788, 21)
games_df_clean.shape = (688, 21)
=== 3. collect_lineups ===
[lineup] 20230711: 20 rows
[lineup] 20230712: 20 rows
[lineup] 20230716: 20 rows
[lineup] 20230717: 20 rows
[lineup] 20230718: 20 rows
[lineup] 20230719: 20 rows
[lineup] 20230720: 20 rows
[lineup] 20230721: 20 rows
[lineup] 20230722: 20 rows
[lineup] 20230723: 20 rows
[lineup] 20230724: 20 rows
[lineup] 20230725: 20 rows
[lineup] 20230726: 20 rows
[lineup] 20230727: 20 rows
[lineup] 20230728: 20 rows
[lineup] 20230729: 20 rows
[lineup] 20230730: 20 rows
[lineup] 20230731: 20 rows
[lineup] 20230732: 20 rows
[lineup] 20230733: 20 rows
[lineup] 20230734: 20 rows
[lineup] 20230735: 20 rows
[lineup] 20230736: 20 rows
[lineup] 20230737: 20 rows
[lineup] 20230738: 20 rows
[lineup] 20230739: 20 rows
[lineup] 20230740: 20 rows
[lineup] 20230001: 20 rows
[lineup] 20230002: 20 rows
[lineup] 20230003: 20 rows
[lineup] 20230004: 20 rows
[lineup] 20230005: 20 rows
[lineup] 202

C:\Users\User\AppData\Local\Temp\ipykernel_27996\728866721.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  hitter_out = pd.concat(hitter_list, ignore_index=True) if hitter_list else pd.DataFrame()
C:\Users\User\AppData\Local\Temp\ipykernel_27996\728866721.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pitcher_out = pd.concat(pitcher_list, ignore_index=True) if pitcher_list else pd.DataFrame()


=== 6. deduplicate season stats (last team + aggregate season) ===
season_hit_df.shape after dedup = (289, 15)
season_pit_df.shape after dedup = (280, 13)
=== 7. clean pitcher season stats ===
=== 8. prepare modeling tables ===
=== 9. build hitter team features ===
hitter_team_features.shape = (1376, 9)
=== 10. build starting pitcher features ===
sp_features.shape = (1376, 13)
sp_features rows per (game_id, team_code):
1    1376
Name: count, dtype: int64
=== 11. make game-level dataset ===
model_df.shape = (688, 70)
game_id duplication check:
count
1    688
Name: count, dtype: int64
Top NA counts in diff features:
sp_ERA_diff        30
sp_FIP_diff        30
sp_WHIP_diff       30
sp_K9_diff         30
sp_BB9_diff        30
sp_HR9_diff        30
sp_KBB_diff        30
lineup_OPS_diff     0
lineup_OBP_diff     0
lineup_SLG_diff     0
dtype: int64


,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,top5_OPS_diff,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,y_home_win
0,20230711,2023-03-23 04:00:00,2023,10001,1001,1003,14674,11404,김동혁,장필준,...,0.015810,0.014867,-0.59,-1.38480,-0.32,0.371,-1.467,-0.702,0.4166,0
1,20230712,2023-03-23 04:00:00,2023,12001,5002,2002,11318,14780,엄상백,강효종,...,0.000434,0.003559,-2.60,-1.42946,-0.55,1.358,-2.648,0.069,1.9023,0
2,20230716,2023-03-24 04:00:00,2023,10001,1001,1003,11379,14613,최원태,허윤동,...,0.024522,0.027390,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,20230717,2023-03-24 04:00:00,2023,12001,5002,2002,15542,13102,슐서,김영준,...,0.007521,-0.003020,-21.38,-8.81277,-4.39,6.342,-24.463,1.087,2.5000,1
4,20230718,2023-03-24 04:00:00,2023,7002,6002,4001,10610,13061,장민재,곽빈,...,-0.009236,-0.007775,1.93,0.87580,0.25,0.465,-1.229,1.070,0.9451,0


(688, 70)


In [ ]:
model_df_23.to_csv("model_df_2023.csv", index=False)

In [ ]:
result_24 = build_modeling_data_pipeline(
    date_list=date_list_24,
    sleep_sec=0.2,
    verbose=True
)

model_df_24 = result_24["model_df"]

display(model_df_24.head())
print(model_df_24.shape)

=== 1. collect_games ===
[games] 2024-03-22: 0 rows
[games] 2024-03-23: 5 rows
[games] 2024-03-24: 5 rows
[games] 2024-03-25: 0 rows
[games] 2024-03-26: 5 rows
[games] 2024-03-27: 5 rows
[games] 2024-03-28: 5 rows
[games] 2024-03-29: 5 rows
[games] 2024-03-30: 5 rows
[games] 2024-03-31: 5 rows
[games] 2024-04-01: 0 rows
[games] 2024-04-02: 5 rows
[games] 2024-04-03: 5 rows
[games] 2024-04-04: 5 rows
[games] 2024-04-05: 5 rows
[games] 2024-04-06: 5 rows
[games] 2024-04-07: 5 rows
[games] 2024-04-08: 0 rows
[games] 2024-04-09: 5 rows
[games] 2024-04-10: 5 rows
[games] 2024-04-11: 5 rows
[games] 2024-04-12: 5 rows
[games] 2024-04-13: 5 rows
[games] 2024-04-14: 5 rows
[games] 2024-04-15: 0 rows
[games] 2024-04-16: 5 rows
[games] 2024-04-17: 5 rows
[games] 2024-04-18: 5 rows
[games] 2024-04-19: 5 rows
[games] 2024-04-20: 5 rows
[games] 2024-04-21: 8 rows
[games] 2024-04-22: 0 rows
[games] 2024-04-23: 5 rows
[games] 2024-04-24: 5 rows
[games] 2024-04-25: 5 rows
[games] 2024-04-26: 5 rows
[ga

C:\Users\User\AppData\Local\Temp\ipykernel_27996\1047996336.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(all_dfs, ignore_index=True).drop_duplicates()


=== 2. filter normal games ===
games_df.shape       = (798, 21)
games_df_clean.shape = (717, 21)
=== 3. collect_lineups ===
[lineup] 20240001: 20 rows
[lineup] 20240002: 20 rows
[lineup] 20240003: 20 rows
[lineup] 20240004: 20 rows
[lineup] 20240005: 20 rows
[lineup] 20240006: 20 rows
[lineup] 20240007: 20 rows
[lineup] 20240008: 20 rows
[lineup] 20240010: 20 rows
[lineup] 20240011: 20 rows
[lineup] 20240012: 20 rows
[lineup] 20240013: 20 rows
[lineup] 20240014: 20 rows
[lineup] 20240015: 20 rows
[lineup] 20240016: 20 rows
[lineup] 20240017: 20 rows
[lineup] 20240018: 20 rows
[lineup] 20240019: 20 rows
[lineup] 20240020: 20 rows
[lineup] 20240021: 20 rows
[lineup] 20240022: 20 rows
[lineup] 20240023: 20 rows
[lineup] 20240026: 20 rows
[lineup] 20240027: 20 rows
[lineup] 20240028: 20 rows
[lineup] 20240029: 20 rows
[lineup] 20240030: 20 rows
[lineup] 20240033: 20 rows
[lineup] 20240031: 20 rows
[lineup] 20240032: 20 rows
[lineup] 20240034: 20 rows
[lineup] 20240035: 20 rows
[lineup] 202

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,top5_OPS_diff,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,y_home_win
0,20240001,2024-03-23 05:00:00,2024,5002,7002,1001,16023,10590,엔스,류현진,...,0.004785,0.019483,0.32,0.36553,-0.05,0.753,0.808,0.177,-0.9509,1
1,20240002,2024-03-23 05:00:00,2024,9002,3001,2001,10126,12295,김광현,윌커슨,...,0.013423,-0.007037,1.09,1.50684,0.24,0.896,2.811,0.507,-4.0756,1
2,20240003,2024-03-23 05:00:00,2024,12001,1001,2002,13942,15860,쿠에바스,코너,...,-0.004067,-0.004753,0.67,-0.18254,0.16,-0.892,0.813,-0.298,-1.3398,0
3,20240004,2024-03-23 05:00:00,2024,2002,10001,6001,16087,15531,크로우,후라도,...,0.082252,0.016166,0.21,-0.78058,0.17,1.604,1.165,-0.452,-1.6980,1
4,20240005,2024-03-23 05:00:00,2024,11001,6002,8005,16065,13941,하트,알칸타라,...,0.063836,0.027098,-2.07,-2.78789,-0.21,5.677,-0.900,-0.768,3.2440,1


(717, 70)


In [ ]:
model_df_24.to_csv("model_df_2024.csv", index=False)

In [ ]:
result_25 = build_modeling_data_pipeline(
    date_list=date_list_25,
    sleep_sec=0.2,
    verbose=True
)

model_df_25 = result_25["model_df"]

display(model_df_25.head())
print(model_df_25.shape)

=== 1. collect_games ===
[games] 2025-03-22: 5 rows
[games] 2025-03-23: 5 rows
[games] 2025-03-24: 0 rows
[games] 2025-03-25: 5 rows
[games] 2025-03-26: 5 rows
[games] 2025-03-27: 5 rows
[games] 2025-03-28: 5 rows
[games] 2025-03-29: 5 rows
[games] 2025-03-30: 5 rows
[games] 2025-03-31: 0 rows
[games] 2025-04-01: 5 rows
[games] 2025-04-02: 5 rows
[games] 2025-04-03: 5 rows
[games] 2025-04-04: 5 rows
[games] 2025-04-05: 5 rows
[games] 2025-04-06: 5 rows
[games] 2025-04-07: 0 rows
[games] 2025-04-08: 5 rows
[games] 2025-04-09: 5 rows
[games] 2025-04-10: 5 rows
[games] 2025-04-11: 5 rows
[games] 2025-04-12: 5 rows
[games] 2025-04-13: 5 rows
[games] 2025-04-14: 0 rows
[games] 2025-04-15: 5 rows
[games] 2025-04-16: 5 rows
[games] 2025-04-17: 5 rows
[games] 2025-04-18: 5 rows
[games] 2025-04-19: 5 rows
[games] 2025-04-20: 5 rows
[games] 2025-04-21: 0 rows
[games] 2025-04-22: 5 rows
[games] 2025-04-23: 5 rows
[games] 2025-04-24: 5 rows
[games] 2025-04-25: 5 rows
[games] 2025-04-26: 5 rows
[ga

C:\Users\User\AppData\Local\Temp\ipykernel_27996\1047996336.py:18: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  out = pd.concat(all_dfs, ignore_index=True).drop_duplicates()


=== 2. filter normal games ===
games_df.shape       = (798, 21)
games_df_clean.shape = (709, 21)
=== 3. collect_lineups ===
[lineup] 20250001: 20 rows
[lineup] 20250002: 20 rows
[lineup] 20250003: 20 rows
[lineup] 20250004: 20 rows
[lineup] 20250005: 20 rows
[lineup] 20250006: 20 rows
[lineup] 20250007: 20 rows
[lineup] 20250008: 20 rows
[lineup] 20250009: 20 rows
[lineup] 20250010: 20 rows
[lineup] 20250011: 20 rows
[lineup] 20250012: 20 rows
[lineup] 20250013: 20 rows
[lineup] 20250014: 20 rows
[lineup] 20250015: 20 rows
[lineup] 20250016: 20 rows
[lineup] 20250017: 20 rows
[lineup] 20250018: 20 rows
[lineup] 20250019: 20 rows
[lineup] 20250020: 20 rows
[lineup] 20250021: 20 rows
[lineup] 20250022: 20 rows
[lineup] 20250023: 20 rows
[lineup] 20250024: 20 rows
[lineup] 20250025: 20 rows
[lineup] 20250026: 20 rows
[lineup] 20250027: 20 rows
[lineup] 20250028: 20 rows
[lineup] 20250029: 20 rows
[lineup] 20250030: 20 rows
[lineup] 20250032: 20 rows
[lineup] 20250033: 20 rows
[lineup] 202

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,top5_OPS_diff,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,y_home_win
0,20250001,2025-03-22 05:00:00,2025,5002,3001,1001,16286,15011,치리노스,반즈,...,0.071262,0.007907,-2.01,-1.07874,-0.22,-0.523,-1.519,-0.534,1.5703,1
1,20250002,2025-03-22 05:00:00,2025,9002,6002,2001,16146,16298,앤더슨,콜어빈,...,-0.046600,0.001351,-2.23,-2.22231,-0.53,4.882,-2.241,0.122,3.1836,1
2,20250003,2025-03-22 05:00:00,2025,1001,10001,7003,15531,16339,후라도,로젠버그,...,0.081965,0.026285,-0.63,0.50802,-0.09,-3.082,-1.345,0.178,0.7444,1
3,20250004,2025-03-22 05:00:00,2025,12001,7002,2002,16138,16313,헤이수스,폰세,...,-0.068278,-0.004885,2.07,1.72832,0.39,-3.481,0.378,0.437,-2.3963,0
4,20250005,2025-03-22 05:00:00,2025,2002,11001,6001,16088,16350,네일,로건,...,-0.010911,-0.005554,-2.28,-1.25967,-0.36,0.574,-1.241,-0.607,1.4834,1


(709, 70)


In [ ]:
model_df_25.to_csv("model_df_2025.csv", index=False)

In [ ]:
date_list_23

['2023-03-22',
 '2023-03-23',
 '2023-03-24',
 '2023-03-25',
 '2023-03-26',
 '2023-03-27',
 '2023-03-28',
 '2023-03-29',
 '2023-03-30',
 '2023-03-31',
 '2023-04-01',
 '2023-04-02',
 '2023-04-03',
 '2023-04-04',
 '2023-04-05',
 '2023-04-06',
 '2023-04-07',
 '2023-04-08',
 '2023-04-09',
 '2023-04-10',
 '2023-04-11',
 '2023-04-12',
 '2023-04-13',
 '2023-04-14',
 '2023-04-15',
 '2023-04-16',
 '2023-04-17',
 '2023-04-18',
 '2023-04-19',
 '2023-04-20',
 '2023-04-21',
 '2023-04-22',
 '2023-04-23',
 '2023-04-24',
 '2023-04-25',
 '2023-04-26',
 '2023-04-27',
 '2023-04-28',
 '2023-04-29',
 '2023-04-30',
 '2023-05-01',
 '2023-05-02',
 '2023-05-03',
 '2023-05-04',
 '2023-05-05',
 '2023-05-06',
 '2023-05-07',
 '2023-05-08',
 '2023-05-09',
 '2023-05-10',
 '2023-05-11',
 '2023-05-12',
 '2023-05-13',
 '2023-05-14',
 '2023-05-15',
 '2023-05-16',
 '2023-05-17',
 '2023-05-18',
 '2023-05-19',
 '2023-05-20',
 '2023-05-21',
 '2023-05-22',
 '2023-05-23',
 '2023-05-24',
 '2023-05-25',
 '2023-05-26',
 '2023-05-

In [ ]:
# save as csv
"""
games_df.to_csv("games.csv", index=False, encoding="utf-8-sig")
lineup_df.to_csv("lineup.csv", index=False, encoding="utf-8-sig")
roster_df.to_csv("roster.csv", index=False, encoding="utf-8-sig")
season_hit_df.to_csv("player_season_hitter.csv", index=False, encoding="utf-8-sig")
season_pit_df.to_csv("player_season_pitcher.csv", index=False, encoding="utf-8-sig")
day_hit_df.to_csv("player_day_hitter.csv", index=False, encoding="utf-8-sig")
day_pit_df.to_csv("player_day_pitcher.csv", index=False, encoding="utf-8-sig")

"""

In [ ]:
def build_prediction_data_for_date(
    date_str,
    team_codes=None,
    roster_code="1",
    sleep_sec=0.2,
    verbose=True
):
    """
    특정 날짜(date_str)의 경기들을 불러와서
    모델 예측에 바로 사용할 수 있는 pred_df를 생성한다.

    Parameters
    ----------
    date_str : str
        예: "2026-04-05"

    team_codes : list[str], optional
        팀코드 리스트. None이면 KBO 10개 구단 기본값 사용.

    roster_code : str, default "1"
        playerRoster 호출용 code 파라미터

    sleep_sec : float, default 0.2
        반복 API 호출 간 텀

    verbose : bool, default True
        진행 로그 출력 여부

    Returns
    -------
    result : dict
        {
            "games_df": ...,
            "games_pred": ...,
            "lineup_df": ...,
            "roster_df": ...,
            "season_hit_df": ...,
            "season_pit_df": ...,
            "lineup_model": ...,
            "season_hit_model": ...,
            "season_pit_model": ...,
            "hitter_team_features": ...,
            "sp_features": ...,
            "pred_df": ...
        }
    """

    # --------------------------------------------------
    # 0) 기본 팀코드
    # --------------------------------------------------
    if team_codes is None:
        team_codes = ["1001", "2002", "3001", "5002", "6002", "7002", "9002", "10001", "11001", "12001"]
        # 삼성, KIA, 롯데, LG, 두산, 한화, SSG, 키움, NC, KT

    # --------------------------------------------------
    # 1) 해당 날짜 경기 수집
    # --------------------------------------------------
    if verbose:
        print("=== 1. collect_games ===")

    games_df = collect_games([date_str], sleep_sec=sleep_sec)

    if verbose:
        print(f"games_df.shape = {games_df.shape}")
        display(games_df.head())

    # --------------------------------------------------
    # 2) 예측 대상으로 사용할 경기 필터
    # --------------------------------------------------
    # 주의:
    # - 학습용 pipeline에서는 game_state == 3 & 점수 존재 조건을 썼음
    # - 만약 미래 경기 예측용이라면 점수 조건은 빼야 함, game_state != 4 활용
    # --------------------------------------------------
    if verbose:
        print("=== 2. filter target games ===")

    games_pred = games_df[
        (games_df["game_state"] != 4)
    ].copy()

    if verbose:
        print(f"games_pred.shape = {games_pred.shape}")
        display(games_pred.head())

    if games_pred.empty:
        raise ValueError("예측 대상으로 사용할 경기가 없습니다. game_state를 확인해봐.")

    # --------------------------------------------------
    # 3) 라인업 수집
    # --------------------------------------------------
    if verbose:
        print("=== 3. collect_lineups ===")

    lineup_df = collect_lineups(games_pred, sleep_sec=sleep_sec)

    if verbose:
        print(f"lineup_df.shape = {lineup_df.shape}")

    # --------------------------------------------------
    # 4) roster 수집
    # --------------------------------------------------
    if verbose:
        print("=== 4. collect_rosters ===")

    roster_df = collect_rosters(
        date=date_str,
        team_codes=team_codes,
        code=roster_code,
        sleep_sec=sleep_sec
    )

    if verbose:
        print(f"roster_df.shape = {roster_df.shape}")

    if roster_df.empty:
        raise ValueError("roster_df가 비어 있습니다. playerSeason 수집이 불가능합니다.")

    # --------------------------------------------------
    # 5) player ids 추출
    # --------------------------------------------------
    player_ids = roster_df["player_id"].astype(str).unique().tolist()

    if verbose:
        print(f"n_unique_player_ids = {len(player_ids)}")

    # --------------------------------------------------
    # 6) playerSeason 수집
    # --------------------------------------------------
    if verbose:
        print("=== 5. collect_player_season ===")

    season_hit_df, season_pit_df = collect_player_season(
        player_ids,
        snapshot_date=date_str,
        sleep_sec=sleep_sec
    )

    target_year = pd.to_datetime(date_str).year

    season_hit_df["year"] = pd.to_numeric(season_hit_df["year"], errors="coerce")
    season_pit_df["year"] = pd.to_numeric(season_pit_df["year"], errors="coerce")

    season_hit_df = season_hit_df[season_hit_df["year"] == target_year].copy()
    season_pit_df = season_pit_df[season_pit_df["year"] == target_year].copy()

    # --------------------------------------------------
    # 7) 시즌 중 팀 이동 선수 처리
    # - player_id + year 기준 1행
    # - team_code는 마지막 팀으로 통일
    # --------------------------------------------------
    if verbose:
        print("=== 6. deduplicate season stats ===")

    season_hit_df = deduplicate_hitter_season_last_team(season_hit_df)
    season_pit_df = deduplicate_pitcher_season_last_team(season_pit_df)

    if verbose:
        print(f"season_hit_df.shape = {season_hit_df.shape}")
        print(f"season_pit_df.shape = {season_pit_df.shape}")

    # --------------------------------------------------
    # 8) pitcher 특수값 처리
    # --------------------------------------------------
    if verbose:
        print("=== 7. clean pitcher season stats ===")

    pit_fill_cols = [c for c in ["ERA", "FIP", "WHIP", "KBB"] if c in season_pit_df.columns]
    if pit_fill_cols:
        season_pit_df[pit_fill_cols] = (
            season_pit_df[pit_fill_cols]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(99.99)
        )

    # --------------------------------------------------
    # 9) 전처리
    # --------------------------------------------------
    if verbose:
        print("=== 8. prepare modeling tables ===")

    # 주의:
    # 여기서는 예측용이므로 prepare_games_for_modeling 대신
    # games_pred를 그대로 쓰되 season/home_win만 맞춰도 충분
    games_pred["game_id"] = games_pred["game_id"].astype(str)
    games_pred["home_team_code"] = games_pred["home_team_code"].astype(str)
    games_pred["away_team_code"] = games_pred["away_team_code"].astype(str)

    if "season" not in games_pred.columns:
        games_pred["season"] = pd.to_datetime(games_pred["game_date"]).dt.year

    lineup_model = prepare_lineup(lineup_df)
    season_hit_model = prepare_season_hitters(season_hit_df)
    season_pit_model = prepare_season_pitchers(season_pit_df)

    # lineup에 season 붙이기
    lineup_model = lineup_model.merge(
        games_pred[["game_id", "season"]],
        on="game_id",
        how="left"
    )

    # --------------------------------------------------
    # 10) 팀 타자 feature 생성
    # --------------------------------------------------
    if verbose:
        print("=== 9. build hitter team features ===")

    hitter_team_features = build_hitter_team_features(lineup_model, season_hit_model)

    if verbose:
        print(f"hitter_team_features.shape = {hitter_team_features.shape}")

    # --------------------------------------------------
    # 11) 선발투수 feature 생성
    # --------------------------------------------------
    if verbose:
        print("=== 10. build starting pitcher features ===")

    sp_features = build_starting_pitcher_features(lineup_model, season_pit_model)

    if verbose:
        print(f"sp_features.shape = {sp_features.shape}")

    # --------------------------------------------------
    # 12) 최종 경기 단위 예측 데이터 생성
    # --------------------------------------------------
    if verbose:
        print("=== 11. make prediction dataframe ===")

    pred_df = make_game_level_dataset(
        games_df_clean=games_pred,
        hitter_team_features=hitter_team_features,
        sp_features=sp_features
    )

    if verbose:
        print(f"pred_df.shape = {pred_df.shape}")
        display(pred_df.head())

    return {
        "games_df": games_df,
        "games_pred": games_pred,
        "lineup_df": lineup_df,
        "roster_df": roster_df,
        "season_hit_df": season_hit_df,
        "season_pit_df": season_pit_df,
        "lineup_model": lineup_model,
        "season_hit_model": season_hit_model,
        "season_pit_model": season_pit_model,
        "hitter_team_features": hitter_team_features,
        "sp_features": sp_features,
        "pred_df": pred_df,
    }

In [ ]:
import joblib
feature_cols = joblib.load("feature_cols.pkl")

In [ ]:
# 예측용 데이터 생성
pred_result = build_prediction_data_for_date(
    date_str="2026-04-05",
    sleep_sec=0.2,
    verbose=True
)

pred_df = pred_result["pred_df"]

# 모델에 실제로 넣을 feature matrix
X_pred = pred_df[feature_cols].copy()

display(X_pred.head())
print(X_pred.shape)

=== 1. collect_games ===
[games] 2026-04-05: 5 rows
games_df.shape = (5, 21)


,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_score,home_win,game_state,game_time,weather,temperature,humidity,windDirection,windSpeed,rainprobability
0,20260036,2026-04-05 05:00:00,2026,6002,7002,1001,16299,16107,잭로그,황준서,...,0,1,3,14:00:00,10200,13.0,47,21300,2.1,None
1,20260037,2026-04-05 05:00:00,2026,10001,5002,1003,16452,16404,와일스,톨허스트,...,6,0,3,14:00:00,10100,12.9,50,21300,1.5,None
2,20260038,2026-04-05 05:00:00,2026,12001,1001,2002,16448,13574,보쉴리,오러클린,...,0,1,3,14:00:00,10100,13.2,52,21100,1.6,None
3,20260039,2026-04-05 05:00:00,2026,2002,11001,6001,16261,16454,올러,토다,...,0,1,3,14:00:00,10100,16.7,37,21100,2.7,None
4,20260040,2026-04-05 05:00:00,2026,3001,9002,8001,11310,16446,박세웅,베니지아노,...,4,0,3,14:00:00,10100,17.2,59,20900,3.2,None


=== 2. filter target games ===
games_pred.shape = (5, 21)


,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,away_score,home_win,game_state,game_time,weather,temperature,humidity,windDirection,windSpeed,rainprobability
0,20260036,2026-04-05 05:00:00,2026,6002,7002,1001,16299,16107,잭로그,황준서,...,0,1,3,14:00:00,10200,13.0,47,21300,2.1,None
1,20260037,2026-04-05 05:00:00,2026,10001,5002,1003,16452,16404,와일스,톨허스트,...,6,0,3,14:00:00,10100,12.9,50,21300,1.5,None
2,20260038,2026-04-05 05:00:00,2026,12001,1001,2002,16448,13574,보쉴리,오러클린,...,0,1,3,14:00:00,10100,13.2,52,21100,1.6,None
3,20260039,2026-04-05 05:00:00,2026,2002,11001,6001,16261,16454,올러,토다,...,0,1,3,14:00:00,10100,16.7,37,21100,2.7,None
4,20260040,2026-04-05 05:00:00,2026,3001,9002,8001,11310,16446,박세웅,베니지아노,...,4,0,3,14:00:00,10100,17.2,59,20900,3.2,None


=== 3. collect_lineups ===
[lineup] 20260036: 20 rows
[lineup] 20260037: 20 rows
[lineup] 20260038: 20 rows
[lineup] 20260039: 20 rows
[lineup] 20260040: 20 rows
lineup_df.shape = (100, 13)
=== 4. collect_rosters ===
[roster] 2026-04-05 team=1001: 29 rows
[roster] 2026-04-05 team=2002: 29 rows
[roster] 2026-04-05 team=3001: 29 rows
[roster] 2026-04-05 team=5002: 29 rows
[roster] 2026-04-05 team=6002: 29 rows
[roster] 2026-04-05 team=7002: 29 rows
[roster] 2026-04-05 team=9002: 29 rows
[roster] 2026-04-05 team=10001: 29 rows
[roster] 2026-04-05 team=11001: 29 rows
[roster] 2026-04-05 team=12001: 29 rows
roster_df.shape = (290, 5)
n_unique_player_ids = 290
=== 5. collect_player_season ===
[playerSeason] 10892: OK
[playerSeason] 10363: OK
[playerSeason] 10640: OK
[playerSeason] 10527: OK
[playerSeason] 10232: OK
[playerSeason] 14116: OK
[playerSeason] 14618: OK
[playerSeason] 12936: OK
[playerSeason] 14798: OK
[playerSeason] 15034: OK
[playerSeason] 15035: OK
[playerSeason] 12934: OK
[pla

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,top5_OPS_diff,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,y_home_win
0,20260036,2026-04-05 05:00:00,2026,6002,7002,1001,16299,16107,잭로그,황준서,...,-0.072501,-0.034328,-2.77,2.10000,-0.30,-6.923,-2.077,0.692,0.1667,1
1,20260037,2026-04-05 05:00:00,2026,10001,5002,1003,16452,16404,와일스,톨허스트,...,-0.104238,-0.042049,-3.09,-0.82424,-0.01,-4.636,-1.364,-1.182,0.5000,0
2,20260038,2026-04-05 05:00:00,2026,12001,1001,2002,16448,13574,보쉴리,오러클린,...,0.237371,0.078335,-5.59,-0.13699,-0.28,-0.084,-0.451,0.000,0.2500,1
3,20260039,2026-04-05 05:00:00,2026,2002,11001,6001,16261,16454,올러,토다,...,-0.167259,-0.064485,-3.27,-1.38322,-0.73,0.629,-3.399,0.000,6.8000,1
4,20260040,2026-04-05 05:00:00,2026,3001,9002,8001,11310,16446,박세웅,베니지아노,...,-0.075523,-0.075791,-1.65,0.35291,-0.15,-0.697,0.987,-0.871,-0.8333,0


,lineup_OPS_diff,lineup_OBP_diff,lineup_SLG_diff,top5_OPS_diff,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff
0,-0.156134,-0.075005,-0.081128,-0.072501,-0.034328,-2.77,2.10000,-0.30,-6.923,-2.077,0.692,0.1667
1,-0.041897,-0.025866,-0.016031,-0.104238,-0.042049,-3.09,-0.82424,-0.01,-4.636,-1.364,-1.182,0.5000
2,0.133775,0.065463,0.068312,0.237371,0.078335,-5.59,-0.13699,-0.28,-0.084,-0.451,0.000,0.2500
3,-0.096762,-0.049168,-0.047594,-0.167259,-0.064485,-3.27,-1.38322,-0.73,0.629,-3.399,0.000,6.8000
4,-0.130325,-0.083252,-0.047073,-0.075523,-0.075791,-1.65,0.35291,-0.15,-0.697,0.987,-0.871,-0.8333


(5, 12)


In [ ]:
pred_df.to_csv("x_pred.csv", index=False, encoding="utf-8-sig")

In [ ]:
pred_df

,game_id,game_date,season,home_team_code,away_team_code,stadium_code,home_sp_id,away_sp_id,home_sp_name,away_sp_name,...,top5_OPS_diff,top5_OBP_diff,sp_ERA_diff,sp_FIP_diff,sp_WHIP_diff,sp_K9_diff,sp_BB9_diff,sp_HR9_diff,sp_KBB_diff,y_home_win
0,20260036,2026-04-05 05:00:00,2026,6002,7002,1001,16299,16107,잭로그,황준서,...,-0.072501,-0.034328,-2.77,2.10000,-0.30,-6.923,-2.077,0.692,0.1667,1
1,20260037,2026-04-05 05:00:00,2026,10001,5002,1003,16452,16404,와일스,톨허스트,...,-0.104238,-0.042049,-3.09,-0.82424,-0.01,-4.636,-1.364,-1.182,0.5000,0
2,20260038,2026-04-05 05:00:00,2026,12001,1001,2002,16448,13574,보쉴리,오러클린,...,0.237371,0.078335,-5.59,-0.13699,-0.28,-0.084,-0.451,0.000,0.2500,1
3,20260039,2026-04-05 05:00:00,2026,2002,11001,6001,16261,16454,올러,토다,...,-0.167259,-0.064485,-3.27,-1.38322,-0.73,0.629,-3.399,0.000,6.8000,1
4,20260040,2026-04-05 05:00:00,2026,3001,9002,8001,11310,16446,박세웅,베니지아노,...,-0.075523,-0.075791,-1.65,0.35291,-0.15,-0.697,0.987,-0.871,-0.8333,0


### 예측 데이터 셋 형성

In [ ]:
def build_prediction_data_for_date(
    date_str,
    team_codes=None,
    roster_code="1",
    sleep_sec=0.2,
    verbose=True
):
    """
    특정 날짜(date_str)의 경기들을 불러와서
    모델 예측에 바로 사용할 수 있는 pred_df를 생성한다.

    Parameters
    ----------
    date_str : str
        예: "2026-04-05"

    team_codes : list[str], optional
        팀코드 리스트. None이면 KBO 10개 구단 기본값 사용.

    roster_code : str, default "1"
        playerRoster 호출용 code 파라미터

    sleep_sec : float, default 0.2
        반복 API 호출 간 텀

    verbose : bool, default True
        진행 로그 출력 여부

    Returns
    -------
    result : dict
        {
            "games_df": ...,
            "games_pred": ...,
            "lineup_df": ...,
            "roster_df": ...,
            "season_hit_df": ...,
            "season_pit_df": ...,
            "lineup_model": ...,
            "season_hit_model": ...,
            "season_pit_model": ...,
            "hitter_team_features": ...,
            "sp_features": ...,
            "pred_df": ...
        }
    """

    # --------------------------------------------------
    # 0) 기본 팀코드
    # --------------------------------------------------
    if team_codes is None:
        team_codes = ["1001", "2002", "3001", "5002", "6002", "7002", "9002", "10001", "11001", "12001"]
        # 삼성, KIA, 롯데, LG, 두산, 한화, SSG, 키움, NC, KT

    # --------------------------------------------------
    # 1) 해당 날짜 경기 수집
    # --------------------------------------------------
    if verbose:
        print("=== 1. collect_games ===")

    games_df = ut.collect_games([date_str], sleep_sec=sleep_sec)

    if verbose:
        print(f"games_df.shape = {games_df.shape}")
        display(games_df.head())

    # --------------------------------------------------
    # 2) 예측 대상으로 사용할 경기 필터
    # --------------------------------------------------
    # 주의:
    # - 학습용 pipeline에서는 game_state == 3 & 점수 존재 조건을 썼음
    # - 여기서는 "어제 완료된 경기"라서 동일하게 써도 됨
    # - 만약 미래 경기 예측용이라면 점수 조건은 빼야 함. game_state != 4 활용
    # --------------------------------------------------
    if verbose:
        print("=== 2. filter target games ===")

    games_pred = games_df[
        (games_df["game_state"] != 4)
    ].copy()

    if verbose:
        print(f"games_pred.shape = {games_pred.shape}")
        display(games_pred.head())

    if games_pred.empty:
        raise ValueError("예측 대상으로 사용할 경기가 없습니다. game_state를 확인해봐.")

    # --------------------------------------------------
    # 3) 라인업 수집
    # --------------------------------------------------
    if verbose:
        print("=== 3. collect_lineups ===")

    lineup_df = ut.collect_lineups(games_pred, sleep_sec=sleep_sec)

    if verbose:
        print(f"lineup_df.shape = {lineup_df.shape}")

    # --------------------------------------------------
    # 4) roster 수집
    # --------------------------------------------------
    if verbose:
        print("=== 4. collect_rosters ===")

    roster_df = ut.collect_rosters(
        date=date_str,
        team_codes=team_codes,
        code=roster_code,
        sleep_sec=sleep_sec
    )

    if verbose:
        print(f"roster_df.shape = {roster_df.shape}")

    if roster_df.empty:
        raise ValueError("roster_df가 비어 있습니다. playerSeason 수집이 불가능합니다.")

    # --------------------------------------------------
    # 5) player ids 추출
    # --------------------------------------------------
    player_ids = roster_df["player_id"].astype(str).unique().tolist()

    if verbose:
        print(f"n_unique_player_ids = {len(player_ids)}")

    # --------------------------------------------------
    # 6) playerSeason 수집
    # --------------------------------------------------
    if verbose:
        print("=== 5. collect_player_season ===")

    season_hit_df, season_pit_df = ut.collect_player_season(
        player_ids,
        snapshot_date=date_str,
        sleep_sec=sleep_sec
    )

    target_year = pd.to_datetime(date_str).year

    season_hit_df["year"] = pd.to_numeric(season_hit_df["year"], errors="coerce")
    season_pit_df["year"] = pd.to_numeric(season_pit_df["year"], errors="coerce")

    season_hit_df = season_hit_df[season_hit_df["year"] == target_year].copy()
    season_pit_df = season_pit_df[season_pit_df["year"] == target_year].copy()

    # --------------------------------------------------
    # 7) 시즌 중 팀 이동 선수 처리
    # - player_id + year 기준 1행
    # - team_code는 마지막 팀으로 통일
    # --------------------------------------------------
    if verbose:
        print("=== 6. deduplicate season stats ===")

    season_hit_df = ut.deduplicate_hitter_season_last_team(season_hit_df)
    season_pit_df = ut.deduplicate_pitcher_season_last_team(season_pit_df)

    if verbose:
        print(f"season_hit_df.shape = {season_hit_df.shape}")
        print(f"season_pit_df.shape = {season_pit_df.shape}")

    # --------------------------------------------------
    # 8) pitcher 특수값 처리
    # --------------------------------------------------
    if verbose:
        print("=== 7. clean pitcher season stats ===")

    pit_fill_cols = [c for c in ["ERA", "FIP", "WHIP", "KBB"] if c in season_pit_df.columns]
    if pit_fill_cols:
        season_pit_df[pit_fill_cols] = (
            season_pit_df[pit_fill_cols]
            .replace([np.inf, -np.inf], np.nan)
            .fillna(99.99)
        )

    # --------------------------------------------------
    # 9) 전처리
    # --------------------------------------------------
    if verbose:
        print("=== 8. prepare modeling tables ===")

    # 주의:
    # 여기서는 예측용이므로 prepare_games_for_modeling 대신
    # games_pred를 그대로 쓰되 season/home_win만 맞춰도 충분
    games_pred["game_id"] = games_pred["game_id"].astype(str)
    games_pred["home_team_code"] = games_pred["home_team_code"].astype(str)
    games_pred["away_team_code"] = games_pred["away_team_code"].astype(str)

    if "season" not in games_pred.columns:
        games_pred["season"] = pd.to_datetime(games_pred["game_date"]).dt.year

    lineup_model = ut.prepare_lineup(lineup_df)
    season_hit_model = ut.prepare_season_hitters(season_hit_df)
    season_pit_model = ut.prepare_season_pitchers(season_pit_df)

    # lineup에 season 붙이기
    lineup_model = lineup_model.merge(
        games_pred[["game_id", "season"]],
        on="game_id",
        how="left"
    )

    # --------------------------------------------------
    # 10) 팀 타자 feature 생성
    # --------------------------------------------------
    if verbose:
        print("=== 9. build hitter team features ===")

    hitter_team_features = ut.build_hitter_team_features(lineup_model, season_hit_model)

    if verbose:
        print(f"hitter_team_features.shape = {hitter_team_features.shape}")

    # --------------------------------------------------
    # 11) 선발투수 feature 생성
    # --------------------------------------------------
    if verbose:
        print("=== 10. build starting pitcher features ===")

    sp_features = ut.build_starting_pitcher_features(lineup_model, season_pit_model)

    if verbose:
        print(f"sp_features.shape = {sp_features.shape}")

    # --------------------------------------------------
    # 12) 최종 경기 단위 예측 데이터 생성
    # --------------------------------------------------
    if verbose:
        print("=== 11. make prediction dataframe ===")

    pred_df = ut.make_game_level_dataset(
        games_df_clean=games_pred,
        hitter_team_features=hitter_team_features,
        sp_features=sp_features
    )

    if verbose:
        print(f"pred_df.shape = {pred_df.shape}")
        display(pred_df.head())

    return {
        "games_df": games_df,
        "games_pred": games_pred,
        "lineup_df": lineup_df,
        "roster_df": roster_df,
        "season_hit_df": season_hit_df,
        "season_pit_df": season_pit_df,
        "lineup_model": lineup_model,
        "season_hit_model": season_hit_model,
        "season_pit_model": season_pit_model,
        "hitter_team_features": hitter_team_features,
        "sp_features": sp_features,
        "pred_df": pred_df,
    }

In [ ]:
import pandas as pd
import numpy as np

def build_season_pit_df_from_model_dfs(df_dict: dict[int, pd.DataFrame]) -> pd.DataFrame:
    """
    model_df_2023/2024/2025 같은 경기 단위 데이터에서
    시즌별 선발투수 요약 테이블(season_pit_df) 생성

    Parameters
    ----------
    df_dict : dict[int, pd.DataFrame]
        예: {2023: model_df_2023, 2024: model_df_2024, 2025: model_df_2025}

    Returns
    -------
    pd.DataFrame
        columns:
        ['year', 'player_id', 'player_name', 'ERA', 'FIP', 'WHIP',
         'K9', 'BB9', 'HR9', 'KBB', 'QS', 'GS']
    """

    all_pitchers = []

    for year, df in df_dict.items():
        tmp = df.copy()

        # home
        home_cols = [
            "home_sp_player_id", "home_sp_player_name",
            "home_sp_ERA", "home_sp_FIP", "home_sp_WHIP",
            "home_sp_K9", "home_sp_BB9", "home_sp_HR9", "home_sp_KBB",
            "home_sp_QS", "home_sp_GS"
        ]
        home = tmp[home_cols].copy()
        home.columns = [
            "player_id", "player_name",
            "ERA", "FIP", "WHIP",
            "K9", "BB9", "HR9", "KBB",
            "QS", "GS"
        ]
        home["year"] = year

        # away
        away_cols = [
            "away_sp_player_id", "away_sp_player_name",
            "away_sp_ERA", "away_sp_FIP", "away_sp_WHIP",
            "away_sp_K9", "away_sp_BB9", "away_sp_HR9", "away_sp_KBB",
            "away_sp_QS", "away_sp_GS"
        ]
        away = tmp[away_cols].copy()
        away.columns = [
            "player_id", "player_name",
            "ERA", "FIP", "WHIP",
            "K9", "BB9", "HR9", "KBB",
            "QS", "GS"
        ]
        away["year"] = year

        season_pitchers = pd.concat([home, away], ignore_index=True)

        # 타입 정리
        season_pitchers["year"] = pd.to_numeric(season_pitchers["year"], errors="coerce")
        season_pitchers["player_id"] = season_pitchers["player_id"].astype(str)

        num_cols = ["ERA", "FIP", "WHIP", "K9", "BB9", "HR9", "KBB", "QS", "GS"]
        for c in num_cols:
            season_pitchers[c] = pd.to_numeric(season_pitchers[c], errors="coerce")

        # player_id 없는 행 제거
        season_pitchers = season_pitchers[
            season_pitchers["player_id"].notna() &
            (season_pitchers["player_id"] != "nan")
        ].copy()

        # 같은 시즌, 같은 투수가 여러 경기에서 반복되므로 1행만 남김
        # 혹시 값이 조금이라도 다르면 GS 큰 값 기준 마지막 기록 사용
        season_pitchers = (
            season_pitchers
            .sort_values(["year", "player_id", "GS"])
            .drop_duplicates(subset=["year", "player_id"], keep="last")
        )

        all_pitchers.append(season_pitchers)

    season_pit_df = pd.concat(all_pitchers, ignore_index=True)

    # 보기 좋게 정렬
    season_pit_df = season_pit_df.sort_values(["year", "player_id"]).reset_index(drop=True)

    return season_pit_df

In [ ]:
model_df_2023 = pd.read_csv("model_df_2023.csv")
model_df_2024 = pd.read_csv("model_df_2024.csv")
model_df_2025 = pd.read_csv("model_df_2025.csv")

season_pit_df = build_season_pit_df_from_model_dfs({
    2023: model_df_2023,
    2024: model_df_2024,
    2025: model_df_2025
})

display(season_pit_df.head())
print(season_pit_df.shape)
print(season_pit_df["year"].value_counts().sort_index())

In [ ]:
season_pit_df.to_csv('seaon_pit_df.csv', index=False)

In [1]:
import pandas as pd

def extract_unique_sp(df: pd.DataFrame) -> pd.DataFrame:
    """
    선발투수 ID + 이름 unique하게 추출

    Parameters
    ----------
    df : model_df 형태의 DataFrame

    Returns
    -------
    sp_df : DataFrame (columns: sp_name, sp_id)
    """

    # 홈/원정 각각 추출
    home_sp = df[["home_sp_name", "home_sp_id"]].rename(
        columns={"home_sp_name": "sp_name", "home_sp_id": "sp_id"}
    )

    away_sp = df[["away_sp_name", "away_sp_id"]].rename(
        columns={"away_sp_name": "sp_name", "away_sp_id": "sp_id"}
    )

    # 합치기
    sp_df = pd.concat([home_sp, away_sp], axis=0)

    # 결측 제거
    sp_df = sp_df.dropna(subset=["sp_id"])

    # 중복 제거 (ID 기준)
    sp_df = sp_df.drop_duplicates(subset=["sp_id"])

    # 정렬 (선택)
    sp_df = sp_df.sort_values("sp_id").reset_index(drop=True)

    return sp_df

In [2]:
df_2023 = pd.read_csv("model_df_2023.csv")
df_2024 = pd.read_csv("model_df_2024.csv")
df_2025 = pd.read_csv("model_df_2025.csv")

df_all = pd.concat([df_2023, df_2024, df_2025], axis=0)

sp_df = extract_unique_sp(df_all)

print(sp_df.head())

  sp_name  sp_id
0     양현종  10058
1     김광현  10126
2     박종훈  10131
3     정우람  10156
4     이용찬  10217


In [4]:
sp_df.to_csv('sp_df.csv', index=False)

In [ ]:
sp_df = pd.read_csv("sp_df.csv")

In [9]:
def get_players_by_name(sp_df, names):
    """
    선수명 리스트를 입력하면 해당 행 반환

    Parameters
    ----------
    sp_df : DataFrame (columns: sp_name, sp_id)
    names : list[str] or str

    Returns
    -------
    DataFrame
    """

    # 문자열 하나 들어와도 처리 가능하게
    if isinstance(names, str):
        names = [names]

    result = sp_df[sp_df["sp_name"].isin(names)].copy()

    return result

In [ ]:
# 걍 id를 때려박을까 고민

def get_players_by_name_safe(sp_df, names): # 찾을 수 없는 선수는 NA로, 그리고 입력한 선수명의 순서대로 list로 출력. 수정해야함

    if isinstance(names, str):
        names = [names]

    result = sp_df[sp_df["sp_name"].isin(names)].copy()

    found_names = set(result["sp_name"])
    missing = list(set(names) - found_names)

    if missing:
        print(f"[경고] 다음 선수는 찾을 수 없음: {missing}")

    return result

In [25]:
names_away = ["알칸타라", "에르난데스", "후라도", "김광현", "치리노스"]
names_home = ["곽빈", "화이트", "네일", "류현진", "구창모"]

id_list_away = get_players_by_name_safe(sp_df, names_away)['sp_id'].tolist()
id_list_home = get_players_by_name_safe(sp_df, names_home)['sp_id'].tolist()

print(id_list_away)
print(id_list_home)

[10126, 13941, 15531, 16160, 16286]
[10590, 11353, 13061, 16088, 16324]


In [22]:
names = ['에르난데스', '화이트']
get_players_by_name_safe(sp_df, names)


,sp_name,sp_id
202,에르난데스,16160
215,화이트,16324


In [34]:
import pandas as pd
import numpy as np


def build_prediction_data_for_date_with_manual_sp(
    target_date,
    schedule_df: pd.DataFrame,
    season_pit_df: pd.DataFrame,
    manual_sp_df: pd.DataFrame,
    target_year: int,
    use_prev_year: bool = True,
    return_full: bool = True
) -> pd.DataFrame:
    """
    수동 입력한 선발투수 ID를 사용하여 pred_df 유사 형태의 예측용 데이터 생성.
    타격 라인업, 날씨 정보는 포함하지 않음.

    Parameters
    ----------
    target_date : str or datetime-like
        예측 대상 날짜
    schedule_df : pd.DataFrame
        경기 일정 데이터.
        최소 필요 정보:
        - game_id
        - game_date
        - home_team_code
        - away_team_code
        선택:
        - season
        - stadium_code
    season_pit_df : pd.DataFrame
        투수 시즌 성적 데이터.
        필요 컬럼(이름은 자동 탐색):
        - player_id / p_no / sp_id
        - player_name / name / p_name
        - year / season
        - ERA, FIP, WHIP, K9, BB9, HR9, KBB, QS, GS
    manual_sp_df : pd.DataFrame
        수동 선발투수 입력 데이터
        필수 컬럼:
        - game_id
        - home_sp_id
        - away_sp_id
    target_year : int
        예측 연도
    use_prev_year : bool, default True
        target_year 기록이 없으면 전년도 기록 fallback
    return_full : bool, default True
        True면 pred_df 유사 전체 컬럼 반환
        False면 최소 컬럼만 반환

    Returns
    -------
    pd.DataFrame
    """

    # -----------------------------
    # 0) 내부 유틸
    # -----------------------------
    def pick_col(candidates, cols, label):
        for c in candidates:
            if c in cols:
                return c
        raise ValueError(f"{label} 컬럼을 찾지 못함. 후보: {candidates}")

    def ensure_datetime_col(df, col):
        out = df.copy()
        out[col] = pd.to_datetime(out[col]).dt.date
        return out

    # -----------------------------
    # 1) manual_sp_df 검증
    # -----------------------------
    required_manual_cols = ["game_id", "home_sp_id", "away_sp_id"]
    missing_manual = [c for c in required_manual_cols if c not in manual_sp_df.columns]
    if missing_manual:
        raise ValueError(f"manual_sp_df에 필요한 컬럼이 없음: {missing_manual}")

    if manual_sp_df["game_id"].duplicated().any():
        dup_ids = manual_sp_df.loc[manual_sp_df["game_id"].duplicated(), "game_id"].tolist()
        raise ValueError(f"manual_sp_df에 중복 game_id가 있음: {dup_ids}")

    manual_sp_df = manual_sp_df.copy()

    # -----------------------------
    # 2) schedule_df 컬럼 탐색
    # -----------------------------
    sch = schedule_df.copy()

    game_id_col = pick_col(["game_id", "s_no"], sch.columns, "경기 ID")
    game_date_col = pick_col(["game_date", "date", "gday"], sch.columns, "경기 날짜")
    home_team_col = pick_col(["home_team_code", "home_t_code", "home_code"], sch.columns, "홈팀 코드")
    away_team_col = pick_col(["away_team_code", "away_t_code", "away_code"], sch.columns, "원정팀 코드")

    season_col = None
    for c in ["season", "year"]:
        if c in sch.columns:
            season_col = c
            break

    stadium_col = None
    for c in ["stadium_code", "stadium", "stadium_id"]:
        if c in sch.columns:
            stadium_col = c
            break

    # 날짜 필터
    sch = sch.rename(columns={
        game_id_col: "game_id",
        game_date_col: "game_date",
        home_team_col: "home_team_code",
        away_team_col: "away_team_code"
    })

    if season_col is not None:
        sch = sch.rename(columns={season_col: "season"})
    else:
        sch["season"] = target_year

    if stadium_col is not None:
        sch = sch.rename(columns={stadium_col: "stadium_code"})
    else:
        sch["stadium_code"] = np.nan

    sch = ensure_datetime_col(sch, "game_date")
    target_date = pd.to_datetime(target_date).date()

    sch = sch.loc[sch["game_date"] == target_date].copy()

    if len(sch) == 0:
        raise ValueError(f"{target_date}에 해당하는 경기 일정이 schedule_df에 없음")

    # -----------------------------
    # 3) 해당 날짜 일정 + manual starter merge
    # -----------------------------
    pred_df = sch.merge(
        manual_sp_df[["game_id", "home_sp_id", "away_sp_id"]],
        on="game_id",
        how="inner"
    ).copy()

    if len(pred_df) == 0:
        raise ValueError("해당 날짜 일정과 manual_sp_df가 매칭되는 game_id가 없음")

    # -----------------------------
    # 4) season_pit_df 컬럼 탐색 및 정리
    # -----------------------------
    pit = season_pit_df.copy()

    pit_id_col = pick_col(["player_id", "p_no", "sp_id"], pit.columns, "투수 ID")
    pit_name_col = pick_col(["player_name", "name", "p_name"], pit.columns, "투수명")
    pit_year_col = pick_col(["year", "season"], pit.columns, "연도")

    stat_cols = ["ERA", "FIP", "WHIP", "K9", "BB9", "HR9", "KBB", "QS", "GS"]
    missing_stats = [c for c in stat_cols if c not in pit.columns]
    if missing_stats:
        raise ValueError(f"season_pit_df에 필요한 지표 컬럼이 없음: {missing_stats}")

    pit = pit[[pit_id_col, pit_name_col, pit_year_col] + stat_cols].copy()

    for c in stat_cols:
        pit[c] = pd.to_numeric(pit[c], errors="coerce")

    # -----------------------------
    # 5) target_year / prev_year / 리그 평균 준비
    # -----------------------------
    pit_this = pit.loc[pit[pit_year_col] == target_year].copy()
    pit_prev = pit.loc[pit[pit_year_col] == (target_year - 1)].copy() if use_prev_year else pit.iloc[0:0].copy()

    if len(pit_prev) > 0:
        league_means = pit_prev[stat_cols].mean()
    elif len(pit_this) > 0:
        league_means = pit_this[stat_cols].mean()
    else:
        league_means = pit[stat_cols].mean()

    pit_this_map = pit_this.drop_duplicates(subset=[pit_id_col]).set_index(pit_id_col) if len(pit_this) > 0 else pd.DataFrame()
    pit_prev_map = pit_prev.drop_duplicates(subset=[pit_id_col]).set_index(pit_id_col) if len(pit_prev) > 0 else pd.DataFrame()

    def get_pitcher_row(pid):
        # 올해 기록 우선
        if len(pit_this_map) > 0 and pid in pit_this_map.index:
            row = pit_this_map.loc[pid]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            return row.to_dict()

        # 전년도 fallback
        if len(pit_prev_map) > 0 and pid in pit_prev_map.index:
            row = pit_prev_map.loc[pid]
            if isinstance(row, pd.DataFrame):
                row = row.iloc[0]
            return row.to_dict()

        # 없으면 리그 평균
        return {
            pit_name_col: np.nan,
            **{c: league_means[c] for c in stat_cols}
        }

    # -----------------------------
    # 6) 홈/원정 선발 정보 생성
    # -----------------------------
    home_rows = pred_df["home_sp_id"].apply(get_pitcher_row)
    away_rows = pred_df["away_sp_id"].apply(get_pitcher_row)

    home_pitcher_df = pd.DataFrame(list(home_rows))
    away_pitcher_df = pd.DataFrame(list(away_rows))

    pred_df["home_sp_player_id"] = pred_df["home_sp_id"]
    pred_df["away_sp_player_id"] = pred_df["away_sp_id"]

    pred_df["home_sp_player_name"] = home_pitcher_df[pit_name_col]
    pred_df["away_sp_player_name"] = away_pitcher_df[pit_name_col]

    pred_df["home_sp_name"] = pred_df["home_sp_player_name"]
    pred_df["away_sp_name"] = pred_df["away_sp_player_name"]

    for c in stat_cols:
        pred_df[f"home_sp_{c}"] = home_pitcher_df[c]
        pred_df[f"away_sp_{c}"] = away_pitcher_df[c]

    # -----------------------------
    # 7) diff 계산
    # -----------------------------
    diff_map = {
        "ERA": "sp_ERA_diff",
        "FIP": "sp_FIP_diff",
        "WHIP": "sp_WHIP_diff",
        "K9": "sp_K9_diff",
        "BB9": "sp_BB9_diff",
        "HR9": "sp_HR9_diff",
        "KBB": "sp_KBB_diff"
    }

    for raw_col, diff_col in diff_map.items():
        pred_df[diff_col] = pred_df[f"home_sp_{raw_col}"] - pred_df[f"away_sp_{raw_col}"]

    # -----------------------------
    # 8) pred_df 유사 컬럼 구성
    #    (라인업/날씨 없음)
    # -----------------------------
    pred_df["home_score"] = np.nan
    pred_df["away_score"] = np.nan
    pred_df["home_win"] = np.nan
    pred_df["game_state"] = np.nan
    pred_df["game_time"] = np.nan

    full_cols = [
        "game_id", "game_date", "season",
        "home_team_code", "away_team_code", "stadium_code",
        "home_sp_id", "away_sp_id",
        "home_sp_name", "away_sp_name",
        "home_score", "away_score", "home_win",
        "game_state", "game_time",
        "home_sp_player_id", "home_sp_player_name",
        "home_sp_ERA", "home_sp_FIP", "home_sp_WHIP",
        "home_sp_K9", "home_sp_BB9", "home_sp_HR9",
        "home_sp_KBB", "home_sp_QS", "home_sp_GS",
        "away_sp_player_id", "away_sp_player_name",
        "away_sp_ERA", "away_sp_FIP", "away_sp_WHIP",
        "away_sp_K9", "away_sp_BB9", "away_sp_HR9",
        "away_sp_KBB", "away_sp_QS", "away_sp_GS",
        "sp_ERA_diff", "sp_FIP_diff", "sp_WHIP_diff",
        "sp_K9_diff", "sp_BB9_diff", "sp_HR9_diff", "sp_KBB_diff"
    ]

    minimal_cols = [
        "game_id", "game_date", "season",
        "home_team_code", "away_team_code", "stadium_code",
        "home_sp_id", "away_sp_id",
        "home_sp_name", "away_sp_name",
        "sp_ERA_diff", "sp_FIP_diff", "sp_WHIP_diff",
        "sp_K9_diff", "sp_BB9_diff", "sp_HR9_diff", "sp_KBB_diff"
    ]

    if return_full:
        pred_df = pred_df[[c for c in full_cols if c in pred_df.columns]].copy()
    else:
        pred_df = pred_df[[c for c in minimal_cols if c in pred_df.columns]].copy()

    return pred_df

In [47]:
import pandas as pd
import json
import urllib.request
import time
import utils as ut

import pandas as pd
import json
import urllib.request
import time

def get_schedule_df(date_str, API_KEY, SECRET):
    """
    date_str: 'YYYY-MM-DD'
    반환: 해당 날짜 경기 일정 DataFrame
    """

    METHOD = "GET"
    PATH = "prediction/gameSchedule"

    # API 구조상 date 전체를 넣기보다 year/month/day로 보내는 형태일 수도 있으니
    # 기존에 네 API 명세대로 맞춰 써도 되고,
    # 일단 현재는 date로 호출이 됐으니 그대로 유지
    QUERY = {
        "date": date_str
    }

    normalized = ut.normalize_query(QUERY)
    timestamp = str(int(time.time()))
    payload = f"{METHOD}|{PATH}|{normalized}|{timestamp}"
    signature = ut.make_signature(SECRET, payload)

    url = f"https://api.statiz.co.kr/baseballApi/{PATH}?{normalized}"

    headers = {
        "X-API-KEY": API_KEY,
        "X-TIMESTAMP": timestamp,
        "X-SIGNATURE": signature,
    }

    req = urllib.request.Request(url, method=METHOD, headers=headers)

    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode("utf-8"))

    # 결과 체크
    if data.get("result_cd") not in [1, "1", 100, "100", None]:
        raise ValueError(f"API 오류: result_cd={data.get('result_cd')}, result_msg={data.get('result_msg')}")

    # '2026-04-09' -> '0409'
    date_key = pd.to_datetime(date_str).strftime("%m%d")

    if date_key not in data:
        raise KeyError(f"응답에 날짜 키 {date_key}가 없음. 사용 가능한 키: {list(data.keys())[:10]} ...")

    games = data[date_key]

    if not isinstance(games, list):
        raise ValueError(f"{date_key}의 값이 list가 아님: {type(games)}")

    df = pd.DataFrame(games)

    return df

In [52]:
# 1. schedule 가져오기
schedule_df = get_schedule_df("2026-04-11", API_KEY, SECRET)
print(schedule_df.head())


       s_no  state  leagueType  year  month  day halfYear  week        hm  \
0  20260064      1       10100  2026      4   11        F     6  14:00:00   
1  20260061      1       10100  2026      4   11        F     6  17:00:00   
2  20260062      1       10100  2026      4   11        F     6  17:00:00   
3  20260063      1       10100  2026      4   11        F     6  17:00:00   
4  20260065      1       10100  2026      4   11        F     6  17:00:00   

     gameDate  ...  rainprobability  s_code  awayTeam  awaySP  homeTeam  \
0  1775883600  ...                0    4003      2002       0      7002   
1  1775894400  ...                0    1001      9002       0      5002   
2  1775894400  ...                0    1003      3001       0     10001   
3  1775894400  ...                0    2002      6002       0     12001   
4  1775894400  ...                0    7003     11001       0      1001   

   homeSP  awaySPName  homeSPName  awayScore  homeScore  
0       0        None       

In [54]:
manual_sp_df = pd.DataFrame({
    "game_id": [20260056, 20260057, 20260058, 20260059, 20260060],
    "home_sp_id": id_list_home,
    "away_sp_id": id_list_away,
})

season_pit_df = pd.read_csv('seaon_pit_df.csv')

pred_df_manual = build_prediction_data_for_date_with_manual_sp(
    target_date="2026-04-11",
    schedule_df=schedule_df,
    season_pit_df=season_pit_df,
    manual_sp_df=manual_sp_df,
    target_year=2026,
    use_prev_year=True,
    return_full=True
)

ValueError: 경기 날짜 컬럼을 찾지 못함. 후보: ['game_date', 'date', 'gday']